<a href="https://colab.research.google.com/github/24071a6236-jpg/polymathai/blob/main/TrainQSVC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# ============================================================
# MILESTONE 4 — FRESH START
# Environment setup
# ============================================================

!pip install -q qiskit==2.5.2 qiskit-machine-learning==0.9.1

In [12]:
# ============================================================
# CELL 2 — IMPORTS
# ============================================================

import os
import json
import time
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

print("✅ Basic imports successful")

✅ Basic imports successful


In [13]:
# MILESTONE 4 — CELL 3
# Verify Milestone 3 processed files

import os
import numpy as np

SEEDS = [42, 1337, 2024]
DATASETS = ["NSL-KDD", "ToN-IoT"]

print("Checking Milestone 3 files...\n")

for dataset in DATASETS:
    for seed in SEEDS:
        path = f"milestone3_processed/{dataset}_seed_{seed}.npz"

        if os.path.exists(path):
            data = np.load(path, allow_pickle=True)

            print(f"✅ {dataset} | Seed {seed}")
            print(f"   X_train : {data['X_train'].shape}")
            print(f"   y_train : {data['y_train'].shape}")
            print(f"   X_val   : {data['X_val'].shape}")
            print(f"   y_val   : {data['y_val'].shape}")
            print()
        else:
            print(f"❌ MISSING: {path}")

print("Verification complete.")

Checking Milestone 3 files...

❌ MISSING: milestone3_processed/NSL-KDD_seed_42.npz
❌ MISSING: milestone3_processed/NSL-KDD_seed_1337.npz
❌ MISSING: milestone3_processed/NSL-KDD_seed_2024.npz
❌ MISSING: milestone3_processed/ToN-IoT_seed_42.npz
❌ MISSING: milestone3_processed/ToN-IoT_seed_1337.npz
❌ MISSING: milestone3_processed/ToN-IoT_seed_2024.npz
Verification complete.


In [15]:
# MILESTONE 4 — CELL 4 (CORRECTED)
# Recreate Milestone 3 processed files

import os
import numpy as np
import pandas as pd

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.decomposition import PCA

SEEDS = [42, 1337, 2024]

os.makedirs("milestone3_processed", exist_ok=True)

# ============================================================
# LOAD DATASETS
# ============================================================

print("Loading ToN-IoT...")
toniot = load_dataset("codymlewis/TON_IoT_network")

ton_train = toniot["train"].to_pandas()

print("ToN-IoT:", ton_train.shape)

print("\nLoading NSL-KDD...")
nslkdd = load_dataset("Mireu-Lab/NSL-KDD")

nsl_train = nslkdd["train"].to_pandas()

print("NSL-KDD:", nsl_train.shape)


# ============================================================
# PREPROCESSING FUNCTION
# ============================================================

def preprocess_and_save(df, dataset_name, target_column):

    print("\n" + "=" * 60)
    print(dataset_name)
    print("=" * 60)

    X = df.drop(columns=[target_column]).copy()
    y = df[target_column].copy()

    # --------------------------------------------------------
    # Convert labels to binary
    # 0 = Normal
    # 1 = Attack
    # --------------------------------------------------------

    if dataset_name == "NSL-KDD":

        # NSL-KDD uses "normal" for normal traffic.
        # Every other class is treated as an attack.
        y = (
            y.astype(str)
             .str.lower()
             .eq("normal")
             .astype(int)
        )

        # Convert:
        # normal -> 0
        # attack -> 1
        y = 1 - y

    else:
        # ToN-IoT already uses:
        # 0 = normal
        # 1 = attack
        y = pd.to_numeric(y).astype(int)

    y = y.to_numpy()

    # --------------------------------------------------------
    # Remove ToN-IoT "type" column
    # It directly describes the attack category.
    # Keeping it would cause target leakage.
    # --------------------------------------------------------

    if "type" in X.columns:
        X = X.drop(columns=["type"])

    # --------------------------------------------------------
    # Find categorical columns
    # --------------------------------------------------------

    categorical_columns = X.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    print("Categorical columns:", len(categorical_columns))

    # Convert categorical columns to strings
    for col in categorical_columns:
        X[col] = X[col].astype(str)

    # --------------------------------------------------------
    # Three required seeds
    # --------------------------------------------------------

    for seed in SEEDS:

        print(f"\n---------- Seed {seed} ----------")

        # 70/30 stratified split
        X_train, X_val, y_train, y_val = train_test_split(
            X,
            y,
            test_size=0.30,
            stratify=y,
            random_state=seed
        )

        # ----------------------------------------------------
        # Ordinal Encoding
        # Fit ONLY on training data
        # ----------------------------------------------------

        if categorical_columns:

            encoder = OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            )

            X_train[categorical_columns] = encoder.fit_transform(
                X_train[categorical_columns]
            )

            X_val[categorical_columns] = encoder.transform(
                X_val[categorical_columns]
            )

        # ----------------------------------------------------
        # Convert everything to numeric
        # ----------------------------------------------------

        X_train = X_train.apply(pd.to_numeric, errors="coerce")
        X_val = X_val.apply(pd.to_numeric, errors="coerce")

        # Handle NaN and infinity
        X_train = X_train.replace(
            [np.inf, -np.inf], np.nan
        ).fillna(0)

        X_val = X_val.replace(
            [np.inf, -np.inf], np.nan
        ).fillna(0)

        # ----------------------------------------------------
        # StandardScaler
        # Fit ONLY on training data
        # ----------------------------------------------------

        scaler = StandardScaler()

        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)

        # ----------------------------------------------------
        # PCA → EXACTLY 8 COMPONENTS
        # Fit ONLY on training data
        # ----------------------------------------------------

        pca = PCA(n_components=8)

        X_train_pca = pca.fit_transform(X_train_scaled)
        X_val_pca = pca.transform(X_val_scaled)

        print("X_train:", X_train_pca.shape)
        print("X_val  :", X_val_pca.shape)
        print("y_train:", y_train.shape)
        print("y_val  :", y_val.shape)

        # ----------------------------------------------------
        # Save processed data
        # ----------------------------------------------------

        output_path = (
            f"milestone3_processed/"
            f"{dataset_name}_seed_{seed}.npz"
        )

        np.savez(
            output_path,
            X_train=X_train_pca,
            X_val=X_val_pca,
            y_train=y_train,
            y_val=y_val
        )

        print("✅ Saved:", output_path)


# ============================================================
# PROCESS ToN-IoT
# ============================================================

preprocess_and_save(
    ton_train,
    "ToN-IoT",
    "label"
)


# ============================================================
# PROCESS NSL-KDD
# ============================================================

preprocess_and_save(
    nsl_train,
    "NSL-KDD",
    "class"
)


print("\n" + "=" * 60)
print("🎯 MILESTONE 3 FILE CREATION COMPLETE")
print("=" * 60)

Loading ToN-IoT...
ToN-IoT: (211043, 44)

Loading NSL-KDD...
NSL-KDD: (151165, 42)

ToN-IoT
Categorical columns: 26

---------- Seed 42 ----------
X_train: (147730, 8)
X_val  : (63313, 8)
y_train: (147730,)
y_val  : (63313,)
✅ Saved: milestone3_processed/ToN-IoT_seed_42.npz

---------- Seed 1337 ----------
X_train: (147730, 8)
X_val  : (63313, 8)
y_train: (147730,)
y_val  : (63313,)
✅ Saved: milestone3_processed/ToN-IoT_seed_1337.npz

---------- Seed 2024 ----------
X_train: (147730, 8)
X_val  : (63313, 8)
y_train: (147730,)
y_val  : (63313,)
✅ Saved: milestone3_processed/ToN-IoT_seed_2024.npz

NSL-KDD
Categorical columns: 3

---------- Seed 42 ----------
X_train: (105815, 8)
X_val  : (45350, 8)
y_train: (105815,)
y_val  : (45350,)
✅ Saved: milestone3_processed/NSL-KDD_seed_42.npz

---------- Seed 1337 ----------
X_train: (105815, 8)
X_val  : (45350, 8)
y_train: (105815,)
y_val  : (45350,)
✅ Saved: milestone3_processed/NSL-KDD_seed_1337.npz

---------- Seed 2024 ----------
X_train: (10

In [16]:
# MILESTONE 4 — CELL 5
# Final verification of Milestone 3 files

import os
import numpy as np

SEEDS = [42, 1337, 2024]
DATASETS = ["NSL-KDD", "ToN-IoT"]

print("Checking all processed files...\n")

all_ok = True

for dataset in DATASETS:
    for seed in SEEDS:

        path = f"milestone3_processed/{dataset}_seed_{seed}.npz"

        if not os.path.exists(path):
            print(f"❌ MISSING: {path}")
            all_ok = False
            continue

        data = np.load(path)

        required = ["X_train", "X_val", "y_train", "y_val"]

        if not all(key in data.files for key in required):
            print(f"❌ Missing arrays: {path}")
            all_ok = False
            continue

        print(
            f"✅ {dataset} | Seed {seed} | "
            f"Train {data['X_train'].shape} | "
            f"Val {data['X_val'].shape}"
        )

if all_ok:
    print("\n🎯 ALL 6 PROCESSED FILES VERIFIED.")
else:
    print("\n❌ Verification failed.")

Checking all processed files...

✅ NSL-KDD | Seed 42 | Train (105815, 8) | Val (45350, 8)
✅ NSL-KDD | Seed 1337 | Train (105815, 8) | Val (45350, 8)
✅ NSL-KDD | Seed 2024 | Train (105815, 8) | Val (45350, 8)
✅ ToN-IoT | Seed 42 | Train (147730, 8) | Val (63313, 8)
✅ ToN-IoT | Seed 1337 | Train (147730, 8) | Val (63313, 8)
✅ ToN-IoT | Seed 2024 | Train (147730, 8) | Val (63313, 8)

🎯 ALL 6 PROCESSED FILES VERIFIED.


In [17]:
# MILESTONE 4 — CELL 6
# Load NSL-KDD Seed 42 for QSVC setup

import numpy as np

DATASET = "NSL-KDD"
SEED = 42

path = f"milestone3_processed/{DATASET}_seed_{SEED}.npz"

data = np.load(path)

X_train = data["X_train"]
y_train = data["y_train"]
X_val = data["X_val"]
y_val = data["y_val"]

print("Dataset :", DATASET)
print("Seed    :", SEED)
print()
print("X_train :", X_train.shape)
print("y_train :", y_train.shape)
print("X_val   :", X_val.shape)
print("y_val   :", y_val.shape)
print()
print("Training labels:")
print("  Normal :", np.sum(y_train == 0))
print("  Attack :", np.sum(y_train == 1))
print()
print("Validation labels:")
print("  Normal :", np.sum(y_val == 0))
print("  Attack :", np.sum(y_val == 1))
print()
print("Number of PCA features:", X_train.shape[1])

assert X_train.shape[1] == 8
assert X_val.shape[1] == 8
assert set(np.unique(y_train)).issubset({0, 1})
assert set(np.unique(y_val)).issubset({0, 1})

print("\n✅ QSVC input data verified.")

Dataset : NSL-KDD
Seed    : 42

X_train : (105815, 8)
y_train : (105815,)
X_val   : (45350, 8)
y_val   : (45350,)

Training labels:
  Normal : 56554
  Attack : 49261

Validation labels:
  Normal : 24238
  Attack : 21112

Number of PCA features: 8

✅ QSVC input data verified.


In [18]:
# MILESTONE 4 — CELL 7
# Prepare PCA features for the quantum feature map

from sklearn.preprocessing import MinMaxScaler

# Map each PCA feature to [0, pi]
qsvc_scaler = MinMaxScaler(
    feature_range=(0, np.pi)
)

X_train_qsvc = qsvc_scaler.fit_transform(X_train)
X_val_qsvc = qsvc_scaler.transform(X_val)

print("Original PCA range:")
print("  Train min:", X_train.min())
print("  Train max:", X_train.max())

print("\nQuantum input range:")
print("  Train min:", X_train_qsvc.min())
print("  Train max:", X_train_qsvc.max())

print("\nShapes:")
print("  X_train_qsvc:", X_train_qsvc.shape)
print("  X_val_qsvc  :", X_val_qsvc.shape)

# Verify the mapping
assert X_train_qsvc.shape == X_train.shape
assert X_val_qsvc.shape == X_val.shape
assert X_train_qsvc.min() >= -1e-10
assert X_train_qsvc.max() <= np.pi + 1e-10

print("\n✅ QSVC feature mapping verified.")

Original PCA range:
  Train min: -24.42101708604639
  Train max: 190.85931411199581

Quantum input range:
  Train min: 0.0
  Train max: 3.141592653589793

Shapes:
  X_train_qsvc: (105815, 8)
  X_val_qsvc  : (45350, 8)

✅ QSVC feature mapping verified.


In [19]:
# MILESTONE 4 — CELL 8
# Create the 8-qubit ZZ quantum feature map

from qiskit.circuit.library import zz_feature_map

feature_map = zz_feature_map(
    feature_dimension=8,
    reps=1,
    entanglement="linear"
)

print(feature_map)

print("\nNumber of qubits:", feature_map.num_qubits)
print("Number of parameters:", feature_map.num_parameters)

assert feature_map.num_qubits == 8

print("\n✅ 8-qubit ZZ feature map created successfully.")

   ┌───┐┌───────────┐                                               »
0: ┤ H ├┤ P(2*x[0]) ├──■────────────────────────────────────■───────»
   ├───┤├───────────┤┌─┴─┐┌──────────────────────────────┐┌─┴─┐     »
1: ┤ H ├┤ P(2*x[1]) ├┤ X ├┤ P((-π + x[0])*(-π + x[1])*2) ├┤ X ├──■──»
   ├───┤├───────────┤└───┘└──────────────────────────────┘└───┘┌─┴─┐»
2: ┤ H ├┤ P(2*x[2]) ├──────────────────────────────────────────┤ X ├»
   ├───┤├───────────┤                                          └───┘»
3: ┤ H ├┤ P(2*x[3]) ├───────────────────────────────────────────────»
   ├───┤├───────────┤                                               »
4: ┤ H ├┤ P(2*x[4]) ├───────────────────────────────────────────────»
   ├───┤├───────────┤                                               »
5: ┤ H ├┤ P(2*x[5]) ├───────────────────────────────────────────────»
   ├───┤├───────────┤                                               »
6: ┤ H ├┤ P(2*x[6]) ├───────────────────────────────────────────────»
   ├───┤├───────────

In [21]:
# MILESTONE 4 — CELL 9
# Create the fidelity-based quantum kernel

from qiskit_machine_learning.kernels import FidelityQuantumKernel

quantum_kernel = FidelityQuantumKernel(
    feature_map=feature_map
)

print("Quantum kernel created successfully.")
print("Feature-map qubits:", feature_map.num_qubits)

assert quantum_kernel is not None
assert feature_map.num_qubits == 8

print("\n✅ FidelityQuantumKernel ready for QSVC.")

Quantum kernel created successfully.
Feature-map qubits: 8

✅ FidelityQuantumKernel ready for QSVC.


In [22]:
# MILESTONE 4 — CELL 10
# Create QSVC classifier

from qiskit_machine_learning.algorithms import QSVC

qsvc = QSVC(
    quantum_kernel=quantum_kernel
)

print("QSVC model created successfully.")
print(qsvc)

print("\n✅ QSVC is ready for training.")

QSVC model created successfully.
QSVC(C=1.0, break_ties=False, cache_size=200, class_weight=None, coef0=0.0,
     decision_function_shape='ovr', degree=3, gamma='scale', max_iter=-1,
     probability=False,
     quantum_kernel=<qiskit_machine_learning.kernels.fidelity_quantum_kernel.FidelityQuantumKernel object at 0x7ca82074b890>,
     random_state=None, shrinking=True, tol=0.001, verbose=False)

✅ QSVC is ready for training.


In [23]:
# MILESTONE 4 — CELL 11
# Prepare reproducible stratified sampling for QSVC

from sklearn.model_selection import train_test_split

# IMPORTANT:
# Set this only when the F08-I2 training budget is confirmed.
TRAIN_SAMPLES = None

def make_qsvc_training_subset(X, y, n_samples, seed):
    """
    Create a reproducible, class-balanced training subset.
    The requested n_samples must be even.
    """

    if n_samples is None:
        raise ValueError(
            "TRAIN_SAMPLES is not set. "
            "Set it to the exact F08-I2 training budget before training."
        )

    if n_samples % 2 != 0:
        raise ValueError("TRAIN_SAMPLES must be even.")

    if n_samples > len(y):
        raise ValueError(
            f"TRAIN_SAMPLES={n_samples} exceeds available training data."
        )

    X_subset, _, y_subset, _ = train_test_split(
        X_train_qsvc,
        y_train,
        train_size=n_samples,
        stratify=y_train,
        random_state=seed
    )

    return X_subset, y_subset


print("QSVC sampling function created.")
print("Available training samples:", len(y_train))
print("Available features:", X_train_qsvc.shape[1])
print("Required seeds:", [42, 1337, 2024])
print("\n⚠️ Training budget is intentionally not set yet.")
print("Waiting for the exact F08-I2 sample budget before qsvc.fit().")

QSVC sampling function created.
Available training samples: 105815
Available features: 8
Required seeds: [42, 1337, 2024]

⚠️ Training budget is intentionally not set yet.
Waiting for the exact F08-I2 sample budget before qsvc.fit().


In [24]:
# MILESTONE 4 — CELL 12
# Set the reproducible QSVC training protocol

TRAIN_SAMPLES = 100

print("QSVC training protocol")
print("----------------------")
print("Training samples :", TRAIN_SAMPLES)
print("Features/qubits  :", 8)
print("Seeds            :", [42, 1337, 2024])
print("Datasets         :", ["NSL-KDD", "ToN-IoT"])
print("Classifier       :", "QSVC")
print("Feature map      :", "8-qubit ZZ")
print("Kernel           :", "FidelityQuantumKernel")

print("\n⚠️ This is the computational training budget for the experiment.")
print("The same training-sample rule must be used consistently")
print("when comparing QSVC, VQC and QNN.")

QSVC training protocol
----------------------
Training samples : 100
Features/qubits  : 8
Seeds            : [42, 1337, 2024]
Datasets         : ['NSL-KDD', 'ToN-IoT']
Classifier       : QSVC
Feature map      : 8-qubit ZZ
Kernel           : FidelityQuantumKernel

⚠️ This is the computational training budget for the experiment.
The same training-sample rule must be used consistently
when comparing QSVC, VQC and QNN.


In [25]:
# MILESTONE 4 — CELL 13
# Reproducible QSVC training function

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

SEEDS = [42, 1337, 2024]
TRAIN_SAMPLES = 100

def train_qsvc(X_train, y_train, seed):
    # 1. Stratified training subset
    X_subset, _, y_subset, _ = train_test_split(
        X_train,
        y_train,
        train_size=TRAIN_SAMPLES,
        stratify=y_train,
        random_state=seed
    )

    # 2. Map PCA features to [0, pi]
    angle_scaler = MinMaxScaler(
        feature_range=(0, np.pi)
    )

    X_subset_scaled = angle_scaler.fit_transform(X_subset)

    # 3. Create 8-qubit ZZ feature map
    feature_map = zz_feature_map(
        feature_dimension=8,
        reps=1,
        entanglement="linear"
    )

    # 4. Fidelity quantum kernel
    quantum_kernel = FidelityQuantumKernel(
        feature_map=feature_map
    )

    # 5. QSVC
    model = QSVC(
        quantum_kernel=quantum_kernel
    )

    # 6. Train
    model.fit(X_subset_scaled, y_subset)

    return model, angle_scaler, X_subset_scaled, y_subset


print("✅ QSVC training function created.")
print("Training samples per run :", TRAIN_SAMPLES)
print("Features / qubits        :", 8)
print("Seeds                    :", SEEDS)

✅ QSVC training function created.
Training samples per run : 100
Features / qubits        : 8
Seeds                    : [42, 1337, 2024]


In [6]:
# ============================================================
# MILESTONE 4 — CELL 14
# NSL-KDD Seed 42 — QSVC TRAIN + CLEAN EVALUATION
# ============================================================

import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score

from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC


# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

SEED = 42
TRAIN_SAMPLES = 100
EVAL_SAMPLES = 100
N_QUBITS = 8


# ------------------------------------------------------------
# LOAD NSL-KDD SEED 42
# ------------------------------------------------------------

data = np.load(
    "milestone3_processed/NSL-KDD_seed_42.npz"
)

X_train = data["X_train"]
y_train = data["y_train"]

X_val = data["X_val"]
y_val = data["y_val"]

print("Dataset       : NSL-KDD")
print("Seed          :", SEED)
print("Full train    :", X_train.shape)
print("Validation    :", X_val.shape)


# ------------------------------------------------------------
# SELECT 100 TRAINING SAMPLES
# ------------------------------------------------------------

X_train_qsvc, _, y_train_qsvc, _ = train_test_split(
    X_train,
    y_train,
    train_size=TRAIN_SAMPLES,
    stratify=y_train,
    random_state=SEED
)

print("\nTraining samples :", len(y_train_qsvc))
print("Normal           :", np.sum(y_train_qsvc == 0))
print("Attack           :", np.sum(y_train_qsvc == 1))


# ------------------------------------------------------------
# SCALE TRAINING DATA TO [0, PI]
# ------------------------------------------------------------

scaler = MinMaxScaler(
    feature_range=(0, np.pi)
)

X_train_scaled = scaler.fit_transform(
    X_train_qsvc
)


# ------------------------------------------------------------
# CREATE 8-QUBIT ZZ FEATURE MAP
# ------------------------------------------------------------

feature_map = zz_feature_map(
    feature_dimension=N_QUBITS,
    reps=1,
    entanglement="linear"
)


# ------------------------------------------------------------
# CREATE QUANTUM KERNEL
# ------------------------------------------------------------

quantum_kernel = FidelityQuantumKernel(
    feature_map=feature_map
)


# ------------------------------------------------------------
# CREATE QSVC
# ------------------------------------------------------------

qsvc = QSVC(
    quantum_kernel=quantum_kernel
)


# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------

print("\nStarting QSVC training...")

train_start = time.time()

qsvc.fit(
    X_train_scaled,
    y_train_qsvc
)

train_time = time.time() - train_start

print("\n========== QSVC TRAINING COMPLETE ==========")
print("Training time : {:.2f} minutes".format(
    train_time / 60
))


# ------------------------------------------------------------
# SELECT 100 VALIDATION SAMPLES
# ------------------------------------------------------------

X_val_eval, _, y_val_eval, _ = train_test_split(
    X_val,
    y_val,
    train_size=EVAL_SAMPLES,
    stratify=y_val,
    random_state=SEED
)

print("\nEvaluation samples :", len(y_val_eval))
print("Normal             :", np.sum(y_val_eval == 0))
print("Attack             :", np.sum(y_val_eval == 1))


# ------------------------------------------------------------
# SCALE VALIDATION DATA
# SAME SCALER FIT ON TRAINING DATA
# ------------------------------------------------------------

X_val_scaled = scaler.transform(
    X_val_eval
)


# ------------------------------------------------------------
# PREDICTION
# ------------------------------------------------------------

print("\nStarting clean QSVC prediction...")

predict_start = time.time()

y_pred = qsvc.predict(
    X_val_scaled
)

predict_time = time.time() - predict_start


# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_val_eval,
    y_pred
)

macro_f1 = f1_score(
    y_val_eval,
    y_pred,
    average="macro"
)


# ------------------------------------------------------------
# FINAL RESULT
# ------------------------------------------------------------

print("\n==========================================")
print(" NSL-KDD SEED 42 — QSVC CLEAN RESULT")
print("==========================================")

print("Training samples  :", TRAIN_SAMPLES)
print("Evaluation samples:", EVAL_SAMPLES)
print("Accuracy          :", round(accuracy, 4))
print("Macro-F1          :", round(macro_f1, 4))
print("Training time     : {:.2f} minutes".format(
    train_time / 60
))
print("Prediction time   : {:.2f} minutes".format(
    predict_time / 60
))

print("\n✅ NSL-KDD Seed 42 QSVC completed.")

Dataset       : NSL-KDD
Seed          : 42
Full train    : (105815, 8)
Validation    : (45350, 8)

Training samples : 100
Normal           : 53
Attack           : 47

Starting QSVC training...

========== QSVC TRAINING COMPLETE ==========
Training time : 3.05 minutes

Evaluation samples : 100
Normal             : 53
Attack             : 47

Starting clean QSVC prediction...

 NSL-KDD SEED 42 — QSVC CLEAN RESULT
Training samples  : 100
Evaluation samples: 100
Accuracy          : 0.9
Macro-F1          : 0.898
Training time     : 3.05 minutes
Prediction time   : 5.48 minutes

✅ NSL-KDD Seed 42 QSVC completed.


In [7]:
# MILESTONE 4 — CELL 15
# NSL-KDD Seed 1337 — QSVC TRAIN + CLEAN EVALUATION

import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score

from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC


# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

SEED = 1337
TRAIN_SAMPLES = 100
EVAL_SAMPLES = 100
N_QUBITS = 8


# ------------------------------------------------------------
# LOAD NSL-KDD SEED 1337
# ------------------------------------------------------------

data = np.load(
    "milestone3_processed/NSL-KDD_seed_1337.npz"
)

X_train = data["X_train"]
y_train = data["y_train"]

X_val = data["X_val"]
y_val = data["y_val"]

print("Dataset       : NSL-KDD")
print("Seed          :", SEED)
print("Full train    :", X_train.shape)
print("Validation    :", X_val.shape)


# ------------------------------------------------------------
# SELECT 100 TRAINING SAMPLES
# ------------------------------------------------------------

X_train_qsvc, _, y_train_qsvc, _ = train_test_split(
    X_train,
    y_train,
    train_size=TRAIN_SAMPLES,
    stratify=y_train,
    random_state=SEED
)

print("\nTraining samples :", len(y_train_qsvc))
print("Normal           :", np.sum(y_train_qsvc == 0))
print("Attack           :", np.sum(y_train_qsvc == 1))


# ------------------------------------------------------------
# SCALE TRAINING DATA TO [0, PI]
# ------------------------------------------------------------

scaler = MinMaxScaler(
    feature_range=(0, np.pi)
)

X_train_scaled = scaler.fit_transform(
    X_train_qsvc
)


# ------------------------------------------------------------
# CREATE 8-QUBIT ZZ FEATURE MAP
# ------------------------------------------------------------

feature_map = zz_feature_map(
    feature_dimension=N_QUBITS,
    reps=1,
    entanglement="linear"
)


# ------------------------------------------------------------
# CREATE QUANTUM KERNEL
# ------------------------------------------------------------

quantum_kernel = FidelityQuantumKernel(
    feature_map=feature_map
)


# ------------------------------------------------------------
# CREATE QSVC
# ------------------------------------------------------------

qsvc = QSVC(
    quantum_kernel=quantum_kernel
)


# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------

print("\nStarting QSVC training...")

train_start = time.time()

qsvc.fit(
    X_train_scaled,
    y_train_qsvc
)

train_time = time.time() - train_start

print("\n========== QSVC TRAINING COMPLETE ==========")
print("Training time : {:.2f} minutes".format(
    train_time / 60
))


# ------------------------------------------------------------
# SELECT 100 VALIDATION SAMPLES
# ------------------------------------------------------------

X_val_eval, _, y_val_eval, _ = train_test_split(
    X_val,
    y_val,
    train_size=EVAL_SAMPLES,
    stratify=y_val,
    random_state=SEED
)

print("\nEvaluation samples :", len(y_val_eval))
print("Normal             :", np.sum(y_val_eval == 0))
print("Attack             :", np.sum(y_val_eval == 1))


# ------------------------------------------------------------
# SCALE VALIDATION DATA
# ------------------------------------------------------------

X_val_scaled = scaler.transform(
    X_val_eval
)


# ------------------------------------------------------------
# PREDICTION
# ------------------------------------------------------------

print("\nStarting clean QSVC prediction...")

predict_start = time.time()

y_pred = qsvc.predict(
    X_val_scaled
)

predict_time = time.time() - predict_start


# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_val_eval,
    y_pred
)

macro_f1 = f1_score(
    y_val_eval,
    y_pred,
    average="macro"
)


# ------------------------------------------------------------
# FINAL RESULT
# ------------------------------------------------------------

print("\n==========================================")
print(" NSL-KDD SEED 1337 — QSVC CLEAN RESULT")
print("==========================================")

print("Training samples  :", TRAIN_SAMPLES)
print("Evaluation samples:", EVAL_SAMPLES)
print("Accuracy          :", round(accuracy, 4))
print("Macro-F1          :", round(macro_f1, 4))
print("Training time     : {:.2f} minutes".format(
    train_time / 60
))
print("Prediction time   : {:.2f} minutes".format(
    predict_time / 60
))

print("\n✅ NSL-KDD Seed 1337 QSVC completed.")

Dataset       : NSL-KDD
Seed          : 1337
Full train    : (105815, 8)
Validation    : (45350, 8)

Training samples : 100
Normal           : 53
Attack           : 47

Starting QSVC training...

========== QSVC TRAINING COMPLETE ==========
Training time : 2.76 minutes

Evaluation samples : 100
Normal             : 53
Attack             : 47

Starting clean QSVC prediction...

 NSL-KDD SEED 1337 — QSVC CLEAN RESULT
Training samples  : 100
Evaluation samples: 100
Accuracy          : 0.9
Macro-F1          : 0.8974
Training time     : 2.76 minutes
Prediction time   : 5.42 minutes

✅ NSL-KDD Seed 1337 QSVC completed.


In [8]:
# MILESTONE 4 — CELL 16
# NSL-KDD Seed 2024 — QSVC TRAIN + CLEAN EVALUATION

import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score

from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC


SEED = 2024
TRAIN_SAMPLES = 100
EVAL_SAMPLES = 100
N_QUBITS = 8


# Load processed NSL-KDD Seed 2024
data = np.load(
    "milestone3_processed/NSL-KDD_seed_2024.npz"
)

X_train = data["X_train"]
y_train = data["y_train"]

X_val = data["X_val"]
y_val = data["y_val"]

print("Dataset       : NSL-KDD")
print("Seed          :", SEED)
print("Full train    :", X_train.shape)
print("Validation    :", X_val.shape)


# Select training samples
X_train_qsvc, _, y_train_qsvc, _ = train_test_split(
    X_train,
    y_train,
    train_size=TRAIN_SAMPLES,
    stratify=y_train,
    random_state=SEED
)

print("\nTraining samples :", len(y_train_qsvc))
print("Normal           :", np.sum(y_train_qsvc == 0))
print("Attack           :", np.sum(y_train_qsvc == 1))


# Scale to [0, pi]
scaler = MinMaxScaler(
    feature_range=(0, np.pi)
)

X_train_scaled = scaler.fit_transform(
    X_train_qsvc
)


# 8-qubit ZZ feature map
feature_map = zz_feature_map(
    feature_dimension=N_QUBITS,
    reps=1,
    entanglement="linear"
)


# Fidelity quantum kernel
quantum_kernel = FidelityQuantumKernel(
    feature_map=feature_map
)


# QSVC
qsvc = QSVC(
    quantum_kernel=quantum_kernel
)


# Train
print("\nStarting QSVC training...")

train_start = time.time()

qsvc.fit(
    X_train_scaled,
    y_train_qsvc
)

train_time = time.time() - train_start

print("\n========== QSVC TRAINING COMPLETE ==========")
print("Training time : {:.2f} minutes".format(
    train_time / 60
))


# Select validation samples
X_val_eval, _, y_val_eval, _ = train_test_split(
    X_val,
    y_val,
    train_size=EVAL_SAMPLES,
    stratify=y_val,
    random_state=SEED
)

print("\nEvaluation samples :", len(y_val_eval))
print("Normal             :", np.sum(y_val_eval == 0))
print("Attack             :", np.sum(y_val_eval == 1))


# Apply training scaler
X_val_scaled = scaler.transform(
    X_val_eval
)


# Predict
print("\nStarting clean QSVC prediction...")

predict_start = time.time()

y_pred = qsvc.predict(
    X_val_scaled
)

predict_time = time.time() - predict_start


# Metrics
accuracy = accuracy_score(
    y_val_eval,
    y_pred
)

macro_f1 = f1_score(
    y_val_eval,
    y_pred,
    average="macro"
)


# Result
print("\n==========================================")
print(" NSL-KDD SEED 2024 — QSVC CLEAN RESULT")
print("==========================================")

print("Training samples  :", TRAIN_SAMPLES)
print("Evaluation samples:", EVAL_SAMPLES)
print("Accuracy          :", round(accuracy, 4))
print("Macro-F1          :", round(macro_f1, 4))
print("Training time     : {:.2f} minutes".format(
    train_time / 60
))
print("Prediction time   : {:.2f} minutes".format(
    predict_time / 60
))

print("\n✅ NSL-KDD Seed 2024 QSVC completed.")

Dataset       : NSL-KDD
Seed          : 2024
Full train    : (105815, 8)
Validation    : (45350, 8)

Training samples : 100
Normal           : 53
Attack           : 47

Starting QSVC training...

========== QSVC TRAINING COMPLETE ==========
Training time : 2.81 minutes

Evaluation samples : 100
Normal             : 53
Attack             : 47

Starting clean QSVC prediction...

 NSL-KDD SEED 2024 — QSVC CLEAN RESULT
Training samples  : 100
Evaluation samples: 100
Accuracy          : 0.91
Macro-F1          : 0.9079
Training time     : 2.81 minutes
Prediction time   : 5.41 minutes

✅ NSL-KDD Seed 2024 QSVC completed.


In [9]:
# MILESTONE 4 — ToN-IoT QSVC — Seed 42

import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC
from sklearn.metrics import accuracy_score, f1_score

SEED = 42
TRAIN_SAMPLES = 100
EVAL_SAMPLES = 100

# Load processed data
data = np.load("milestone3_processed/ToN-IoT_seed_42.npz")

X_train = data["X_train"]
y_train = data["y_train"]
X_val = data["X_val"]
y_val = data["y_val"]

print("Dataset       : ToN-IoT")
print("Seed          :", SEED)
print("Full train    :", X_train.shape)
print("Validation    :", X_val.shape)

# -----------------------------
# Select 100 balanced training samples
# -----------------------------
X_subset, _, y_subset, _ = train_test_split(
    X_train,
    y_train,
    train_size=TRAIN_SAMPLES,
    stratify=y_train,
    random_state=SEED
)

# Quantum angle scaling
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_subset_scaled = scaler.fit_transform(X_subset)

# -----------------------------
# QSVC
# -----------------------------
feature_map = zz_feature_map(
    feature_dimension=8,
    reps=1,
    entanglement="linear"
)

quantum_kernel = FidelityQuantumKernel(
    feature_map=feature_map
)

qsvc = QSVC(
    quantum_kernel=quantum_kernel
)

print("\nTraining samples :", len(y_subset))
print("Normal           :", np.sum(y_subset == 0))
print("Attack           :", np.sum(y_subset == 1))

print("\nStarting QSVC training...")
start_train = time.time()

qsvc.fit(X_subset_scaled, y_subset)

training_time = time.time() - start_train

print("\n========== QSVC TRAINING COMPLETE ==========")
print("Training time : {:.2f} minutes".format(training_time / 60))

# -----------------------------
# Select 100 balanced evaluation samples
# -----------------------------
X_eval, _, y_eval, _ = train_test_split(
    X_val,
    y_val,
    train_size=EVAL_SAMPLES,
    stratify=y_val,
    random_state=SEED
)

X_eval_scaled = scaler.transform(X_eval)

print("\nEvaluation samples :", len(y_eval))
print("Normal             :", np.sum(y_eval == 0))
print("Attack             :", np.sum(y_eval == 1))

# -----------------------------
# Prediction
# -----------------------------
print("\nStarting clean QSVC prediction...")
start_pred = time.time()

y_pred = qsvc.predict(X_eval_scaled)

prediction_time = time.time() - start_pred

accuracy = accuracy_score(y_eval, y_pred)
macro_f1 = f1_score(y_eval, y_pred, average="macro")

print("\n==========================================")
print("ToN-IoT SEED 42 — QSVC CLEAN RESULT")
print("==========================================")
print("Training samples  :", len(y_subset))
print("Evaluation samples:", len(y_eval))
print("Accuracy          :", round(accuracy, 4))
print("Macro-F1          :", round(macro_f1, 4))
print("Training time     : {:.2f} minutes".format(training_time / 60))
print("Prediction time   : {:.2f} minutes".format(prediction_time / 60))

print("\n✅ ToN-IoT Seed 42 QSVC completed.")

Dataset       : ToN-IoT
Seed          : 42
Full train    : (147730, 8)
Validation    : (63313, 8)

Training samples : 100
Normal           : 24
Attack           : 76

Starting QSVC training...

========== QSVC TRAINING COMPLETE ==========
Training time : 2.77 minutes

Evaluation samples : 100
Normal             : 24
Attack             : 76

Starting clean QSVC prediction...

ToN-IoT SEED 42 — QSVC CLEAN RESULT
Training samples  : 100
Evaluation samples: 100
Accuracy          : 0.91
Macro-F1          : 0.871
Training time     : 2.77 minutes
Prediction time   : 5.37 minutes

✅ ToN-IoT Seed 42 QSVC completed.


In [10]:
# MILESTONE 4 — ToN-IoT QSVC — Seed 1337

import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

SEED = 1337
TRAIN_SAMPLES = 100
EVAL_SAMPLES = 100

# Load processed data
data = np.load("milestone3_processed/ToN-IoT_seed_1337.npz")

X_train = data["X_train"]
y_train = data["y_train"]
X_val = data["X_val"]
y_val = data["y_val"]

print("Dataset       : ToN-IoT")
print("Seed          :", SEED)
print("Full train    :", X_train.shape)
print("Validation    :", X_val.shape)

# -----------------------------
# Select training samples
# -----------------------------
X_subset, _, y_subset, _ = train_test_split(
    X_train,
    y_train,
    train_size=TRAIN_SAMPLES,
    stratify=y_train,
    random_state=SEED
)

# Map PCA features to quantum angle range [0, pi]
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_subset_scaled = scaler.fit_transform(X_subset)

print("\nTraining samples :", len(y_subset))
print("Normal           :", np.sum(y_subset == 0))
print("Attack           :", np.sum(y_subset == 1))

# -----------------------------
# Create QSVC
# -----------------------------
feature_map = zz_feature_map(
    feature_dimension=8,
    reps=1,
    entanglement="linear"
)

quantum_kernel = FidelityQuantumKernel(
    feature_map=feature_map
)

qsvc = QSVC(
    quantum_kernel=quantum_kernel
)

# -----------------------------
# Training
# -----------------------------
print("\nStarting QSVC training...")
start_train = time.time()

qsvc.fit(X_subset_scaled, y_subset)

training_time = time.time() - start_train

print("\n========== QSVC TRAINING COMPLETE ==========")
print("Training time : {:.2f} minutes".format(training_time / 60))

# -----------------------------
# Select evaluation samples
# -----------------------------
X_eval, _, y_eval, _ = train_test_split(
    X_val,
    y_val,
    train_size=EVAL_SAMPLES,
    stratify=y_val,
    random_state=SEED
)

X_eval_scaled = scaler.transform(X_eval)

print("\nEvaluation samples :", len(y_eval))
print("Normal             :", np.sum(y_eval == 0))
print("Attack             :", np.sum(y_eval == 1))

# -----------------------------
# Clean prediction
# -----------------------------
print("\nStarting clean QSVC prediction...")
start_pred = time.time()

y_pred = qsvc.predict(X_eval_scaled)

prediction_time = time.time() - start_pred

accuracy = accuracy_score(y_eval, y_pred)
macro_f1 = f1_score(y_eval, y_pred, average="macro")

print("\n==========================================")
print("ToN-IoT SEED 1337 — QSVC CLEAN RESULT")
print("==========================================")
print("Training samples  :", len(y_subset))
print("Evaluation samples:", len(y_eval))
print("Accuracy          :", round(accuracy, 4))
print("Macro-F1          :", round(macro_f1, 4))
print("Training time     : {:.2f} minutes".format(training_time / 60))
print("Prediction time   : {:.2f} minutes".format(prediction_time / 60))

print("\n✅ ToN-IoT Seed 1337 QSVC completed.")

Dataset       : ToN-IoT
Seed          : 1337
Full train    : (147730, 8)
Validation    : (63313, 8)

Training samples : 100
Normal           : 24
Attack           : 76

Starting QSVC training...

========== QSVC TRAINING COMPLETE ==========
Training time : 2.73 minutes

Evaluation samples : 100
Normal             : 24
Attack             : 76

Starting clean QSVC prediction...

ToN-IoT SEED 1337 — QSVC CLEAN RESULT
Training samples  : 100
Evaluation samples: 100
Accuracy          : 0.96
Macro-F1          : 0.9417
Training time     : 2.73 minutes
Prediction time   : 5.34 minutes

✅ ToN-IoT Seed 1337 QSVC completed.


In [11]:
# MILESTONE 4 — ToN-IoT QSVC — Seed 2024

import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

SEED = 2024
TRAIN_SAMPLES = 100
EVAL_SAMPLES = 100

# Load processed data
data = np.load("milestone3_processed/ToN-IoT_seed_2024.npz")

X_train = data["X_train"]
y_train = data["y_train"]
X_val = data["X_val"]
y_val = data["y_val"]

print("Dataset       : ToN-IoT")
print("Seed          :", SEED)
print("Full train    :", X_train.shape)
print("Validation    :", X_val.shape)

# -----------------------------
# Select training samples
# -----------------------------
X_subset, _, y_subset, _ = train_test_split(
    X_train,
    y_train,
    train_size=TRAIN_SAMPLES,
    stratify=y_train,
    random_state=SEED
)

# Map PCA features to [0, pi]
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_subset_scaled = scaler.fit_transform(X_subset)

print("\nTraining samples :", len(y_subset))
print("Normal           :", np.sum(y_subset == 0))
print("Attack           :", np.sum(y_subset == 1))

# -----------------------------
# Create QSVC
# -----------------------------
feature_map = zz_feature_map(
    feature_dimension=8,
    reps=1,
    entanglement="linear"
)

quantum_kernel = FidelityQuantumKernel(
    feature_map=feature_map
)

qsvc = QSVC(
    quantum_kernel=quantum_kernel
)

# -----------------------------
# Training
# -----------------------------
print("\nStarting QSVC training...")
start_train = time.time()

qsvc.fit(X_subset_scaled, y_subset)

training_time = time.time() - start_train

print("\n========== QSVC TRAINING COMPLETE ==========")
print("Training time : {:.2f} minutes".format(training_time / 60))

# -----------------------------
# Select evaluation samples
# -----------------------------
X_eval, _, y_eval, _ = train_test_split(
    X_val,
    y_val,
    train_size=EVAL_SAMPLES,
    stratify=y_val,
    random_state=SEED
)

X_eval_scaled = scaler.transform(X_eval)

print("\nEvaluation samples :", len(y_eval))
print("Normal             :", np.sum(y_eval == 0))
print("Attack             :", np.sum(y_eval == 1))

# -----------------------------
# Clean prediction
# -----------------------------
print("\nStarting clean QSVC prediction...")
start_pred = time.time()

y_pred = qsvc.predict(X_eval_scaled)

prediction_time = time.time() - start_pred

accuracy = accuracy_score(y_eval, y_pred)
macro_f1 = f1_score(y_eval, y_pred, average="macro")

print("\n==========================================")
print("ToN-IoT SEED 2024 — QSVC CLEAN RESULT")
print("==========================================")
print("Training samples  :", len(y_subset))
print("Evaluation samples:", len(y_eval))
print("Accuracy          :", round(accuracy, 4))
print("Macro-F1          :", round(macro_f1, 4))
print("Training time     : {:.2f} minutes".format(training_time / 60))
print("Prediction time   : {:.2f} minutes".format(prediction_time / 60))

print("\n✅ ToN-IoT Seed 2024 QSVC completed.")

Dataset       : ToN-IoT
Seed          : 2024
Full train    : (147730, 8)
Validation    : (63313, 8)

Training samples : 100
Normal           : 24
Attack           : 76

Starting QSVC training...

========== QSVC TRAINING COMPLETE ==========
Training time : 2.81 minutes

Evaluation samples : 100
Normal             : 24
Attack             : 76

Starting clean QSVC prediction...

ToN-IoT SEED 2024 — QSVC CLEAN RESULT
Training samples  : 100
Evaluation samples: 100
Accuracy          : 0.93
Macro-F1          : 0.8926
Training time     : 2.81 minutes
Prediction time   : 5.43 minutes

✅ ToN-IoT Seed 2024 QSVC completed.


In [12]:
# MILESTONE 4 — VQC — NSL-KDD — Seed 42

import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score

from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit_machine_learning.algorithms import VQC
from qiskit_machine_learning.optimizers import COBYLA

SEED = 42
TRAIN_SAMPLES = 100
EVAL_SAMPLES = 100

# ============================================================
# 1. Load processed NSL-KDD data
# ============================================================

data = np.load("milestone3_processed/NSL-KDD_seed_42.npz")

X_train = data["X_train"]
y_train = data["y_train"]
X_val = data["X_val"]
y_val = data["y_val"]

print("Dataset       : NSL-KDD")
print("Model         : VQC")
print("Seed          :", SEED)
print("Full train    :", X_train.shape)
print("Validation    :", X_val.shape)

# ============================================================
# 2. Select 100 stratified training samples
# ============================================================

X_subset, _, y_subset, _ = train_test_split(
    X_train,
    y_train,
    train_size=TRAIN_SAMPLES,
    stratify=y_train,
    random_state=SEED
)

# ============================================================
# 3. Scale PCA features to [0, pi]
# ============================================================

scaler = MinMaxScaler(feature_range=(0, np.pi))

X_subset_scaled = scaler.fit_transform(X_subset)

print("\nTraining samples :", len(y_subset))
print("Normal           :", np.sum(y_subset == 0))
print("Attack           :", np.sum(y_subset == 1))

# ============================================================
# 4. Create 8-qubit feature map and ansatz
# ============================================================

feature_map = zz_feature_map(
    feature_dimension=8,
    reps=1,
    entanglement="linear"
)

ansatz = real_amplitudes(
    num_qubits=8,
    reps=1,
    entanglement="linear"
)

print("\nQubits            : 8")
print("Feature-map reps  : 1")
print("Ansatz reps       : 1")

# ============================================================
# 5. Create VQC
# ============================================================

optimizer = COBYLA(
    maxiter=30
)

vqc = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=optimizer
)

# ============================================================
# 6. Train VQC
# ============================================================

print("\nStarting VQC training...")

start_train = time.time()

vqc.fit(
    X_subset_scaled,
    y_subset
)

training_time = time.time() - start_train

print("\n========== VQC TRAINING COMPLETE ==========")
print(
    "Training time : {:.2f} minutes".format(
        training_time / 60
    )
)

# ============================================================
# 7. Select 100 clean evaluation samples
# ============================================================

X_eval, _, y_eval, _ = train_test_split(
    X_val,
    y_val,
    train_size=EVAL_SAMPLES,
    stratify=y_val,
    random_state=SEED
)

X_eval_scaled = scaler.transform(X_eval)

print("\nEvaluation samples :", len(y_eval))
print("Normal             :", np.sum(y_eval == 0))
print("Attack             :", np.sum(y_eval == 1))

# ============================================================
# 8. Clean prediction
# ============================================================

print("\nStarting clean VQC prediction...")

start_pred = time.time()

y_pred = vqc.predict(X_eval_scaled)

prediction_time = time.time() - start_pred

# ============================================================
# 9. Metrics
# ============================================================

accuracy = accuracy_score(
    y_eval,
    y_pred
)

macro_f1 = f1_score(
    y_eval,
    y_pred,
    average="macro"
)

print("\n==========================================")
print("NSL-KDD SEED 42 — VQC CLEAN RESULT")
print("==========================================")

print("Training samples  :", len(y_subset))
print("Evaluation samples:", len(y_eval))
print("Accuracy          :", round(accuracy, 4))
print("Macro-F1          :", round(macro_f1, 4))
print(
    "Training time     : {:.2f} minutes".format(
        training_time / 60
    )
)
print(
    "Prediction time   : {:.2f} minutes".format(
        prediction_time / 60
    )
)

print("\n✅ NSL-KDD Seed 42 VQC completed.")

Dataset       : NSL-KDD
Model         : VQC
Seed          : 42
Full train    : (105815, 8)
Validation    : (45350, 8)

Training samples : 100
Normal           : 53
Attack           : 47

Qubits            : 8
Feature-map reps  : 1
Ansatz reps       : 1

Starting VQC training...

========== VQC TRAINING COMPLETE ==========
Training time : 1.63 minutes

Evaluation samples : 100
Normal             : 53
Attack             : 47

Starting clean VQC prediction...

NSL-KDD SEED 42 — VQC CLEAN RESULT
Training samples  : 100
Evaluation samples: 100
Accuracy          : 0.65
Macro-F1          : 0.6457
Training time     : 1.63 minutes
Prediction time   : 0.07 minutes

✅ NSL-KDD Seed 42 VQC completed.


In [13]:
# MILESTONE 4 — VQC — NSL-KDD — Seed 1337

import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score

from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit_machine_learning.algorithms import VQC
from qiskit_machine_learning.optimizers import COBYLA

SEED = 1337
TRAIN_SAMPLES = 100
EVAL_SAMPLES = 100

# ============================================================
# 1. Load processed NSL-KDD data
# ============================================================

data = np.load(
    "milestone3_processed/NSL-KDD_seed_1337.npz"
)

X_train = data["X_train"]
y_train = data["y_train"]

X_val = data["X_val"]
y_val = data["y_val"]

print("Dataset       : NSL-KDD")
print("Model         : VQC")
print("Seed          :", SEED)
print("Full train    :", X_train.shape)
print("Validation    :", X_val.shape)

# ============================================================
# 2. Stratified training subset
# ============================================================

X_subset, _, y_subset, _ = train_test_split(
    X_train,
    y_train,
    train_size=TRAIN_SAMPLES,
    stratify=y_train,
    random_state=SEED
)

print("\nTraining samples :", len(y_subset))
print("Normal           :", np.sum(y_subset == 0))
print("Attack           :", np.sum(y_subset == 1))

# ============================================================
# 3. Map features to [0, pi]
# ============================================================

scaler = MinMaxScaler(
    feature_range=(0, np.pi)
)

X_subset_scaled = scaler.fit_transform(
    X_subset
)

# ============================================================
# 4. 8-qubit feature map
# ============================================================

feature_map = zz_feature_map(
    feature_dimension=8,
    reps=1,
    entanglement="linear"
)

# ============================================================
# 5. Variational ansatz
# ============================================================

ansatz = real_amplitudes(
    num_qubits=8,
    reps=1,
    entanglement="linear"
)

print("\nQubits            : 8")
print("Feature-map reps  : 1")
print("Ansatz reps       : 1")

# ============================================================
# 6. VQC
# ============================================================

optimizer = COBYLA(
    maxiter=30
)

vqc = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=optimizer
)

# ============================================================
# 7. Train
# ============================================================

print("\nStarting VQC training...")

train_start = time.time()

vqc.fit(
    X_subset_scaled,
    y_subset
)

train_time = time.time() - train_start

print("\n========== VQC TRAINING COMPLETE ==========")
print(
    "Training time : {:.2f} minutes".format(
        train_time / 60
    )
)

# ============================================================
# 8. Select clean evaluation samples
# ============================================================

X_eval, _, y_eval, _ = train_test_split(
    X_val,
    y_val,
    train_size=EVAL_SAMPLES,
    stratify=y_val,
    random_state=SEED
)

X_eval_scaled = scaler.transform(
    X_eval
)

print("\nEvaluation samples :", len(y_eval))
print("Normal             :", np.sum(y_eval == 0))
print("Attack             :", np.sum(y_eval == 1))

# ============================================================
# 9. Clean prediction
# ============================================================

print("\nStarting clean VQC prediction...")

pred_start = time.time()

y_pred = vqc.predict(
    X_eval_scaled
)

pred_time = time.time() - pred_start

# ============================================================
# 10. Metrics
# ============================================================

accuracy = accuracy_score(
    y_eval,
    y_pred
)

macro_f1 = f1_score(
    y_eval,
    y_pred,
    average="macro"
)

print("\n==========================================")
print("NSL-KDD SEED 1337 — VQC CLEAN RESULT")
print("==========================================")

print("Training samples  :", len(y_subset))
print("Evaluation samples:", len(y_eval))
print("Accuracy          :", round(accuracy, 4))
print("Macro-F1          :", round(macro_f1, 4))
print(
    "Training time     : {:.2f} minutes".format(
        train_time / 60
    )
)
print(
    "Prediction time   : {:.2f} minutes".format(
        pred_time / 60
    )
)

print("\n✅ NSL-KDD Seed 1337 VQC completed.")

Dataset       : NSL-KDD
Model         : VQC
Seed          : 1337
Full train    : (105815, 8)
Validation    : (45350, 8)

Training samples : 100
Normal           : 53
Attack           : 47

Qubits            : 8
Feature-map reps  : 1
Ansatz reps       : 1

Starting VQC training...

========== VQC TRAINING COMPLETE ==========
Training time : 1.60 minutes

Evaluation samples : 100
Normal             : 53
Attack             : 47

Starting clean VQC prediction...

NSL-KDD SEED 1337 — VQC CLEAN RESULT
Training samples  : 100
Evaluation samples: 100
Accuracy          : 0.54
Macro-F1          : 0.5066
Training time     : 1.60 minutes
Prediction time   : 0.05 minutes

✅ NSL-KDD Seed 1337 VQC completed.


In [14]:
# MILESTONE 4 — VQC — NSL-KDD — Seed 2024

import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score

from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit_machine_learning.algorithms import VQC
from qiskit_machine_learning.optimizers import COBYLA

SEED = 2024
TRAIN_SAMPLES = 100
EVAL_SAMPLES = 100

# ============================================================
# 1. Load processed NSL-KDD Seed 2024
# ============================================================

data = np.load(
    "milestone3_processed/NSL-KDD_seed_2024.npz"
)

X_train = data["X_train"]
y_train = data["y_train"]

X_val = data["X_val"]
y_val = data["y_val"]

print("Dataset       : NSL-KDD")
print("Model         : VQC")
print("Seed          :", SEED)
print("Full train    :", X_train.shape)
print("Validation    :", X_val.shape)

# ============================================================
# 2. Stratified training subset
# ============================================================

X_subset, _, y_subset, _ = train_test_split(
    X_train,
    y_train,
    train_size=TRAIN_SAMPLES,
    stratify=y_train,
    random_state=SEED
)

print("\nTraining samples :", len(y_subset))
print("Normal           :", np.sum(y_subset == 0))
print("Attack           :", np.sum(y_subset == 1))

# ============================================================
# 3. Scale features to [0, pi]
# ============================================================

scaler = MinMaxScaler(
    feature_range=(0, np.pi)
)

X_subset_scaled = scaler.fit_transform(
    X_subset
)

# ============================================================
# 4. 8-qubit ZZ feature map
# ============================================================

feature_map = zz_feature_map(
    feature_dimension=8,
    reps=1,
    entanglement="linear"
)

# ============================================================
# 5. Variational ansatz
# ============================================================

ansatz = real_amplitudes(
    num_qubits=8,
    reps=1,
    entanglement="linear"
)

print("\nQubits            : 8")
print("Feature-map reps  : 1")
print("Ansatz reps       : 1")

# ============================================================
# 6. VQC
# ============================================================

optimizer = COBYLA(
    maxiter=30
)

vqc = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=optimizer
)

# ============================================================
# 7. Train
# ============================================================

print("\nStarting VQC training...")

train_start = time.time()

vqc.fit(
    X_subset_scaled,
    y_subset
)

train_time = time.time() - train_start

print("\n========== VQC TRAINING COMPLETE ==========")
print(
    "Training time : {:.2f} minutes".format(
        train_time / 60
    )
)

# ============================================================
# 8. Select clean evaluation samples
# ============================================================

X_eval, _, y_eval, _ = train_test_split(
    X_val,
    y_val,
    train_size=EVAL_SAMPLES,
    stratify=y_val,
    random_state=SEED
)

X_eval_scaled = scaler.transform(
    X_eval
)

print("\nEvaluation samples :", len(y_eval))
print("Normal             :", np.sum(y_eval == 0))
print("Attack             :", np.sum(y_eval == 1))

# ============================================================
# 9. Clean prediction
# ============================================================

print("\nStarting clean VQC prediction...")

pred_start = time.time()

y_pred = vqc.predict(
    X_eval_scaled
)

pred_time = time.time() - pred_start

# ============================================================
# 10. Metrics
# ============================================================

accuracy = accuracy_score(
    y_eval,
    y_pred
)

macro_f1 = f1_score(
    y_eval,
    y_pred,
    average="macro"
)

print("\n==========================================")
print("NSL-KDD SEED 2024 — VQC CLEAN RESULT")
print("==========================================")

print("Training samples  :", len(y_subset))
print("Evaluation samples:", len(y_eval))
print("Accuracy          :", round(accuracy, 4))
print("Macro-F1          :", round(macro_f1, 4))
print(
    "Training time     : {:.2f} minutes".format(
        train_time / 60
    )
)
print(
    "Prediction time   : {:.2f} minutes".format(
        pred_time / 60
    )
)

print("\n✅ NSL-KDD Seed 2024 VQC completed.")

Dataset       : NSL-KDD
Model         : VQC
Seed          : 2024
Full train    : (105815, 8)
Validation    : (45350, 8)

Training samples : 100
Normal           : 53
Attack           : 47

Qubits            : 8
Feature-map reps  : 1
Ansatz reps       : 1

Starting VQC training...

========== VQC TRAINING COMPLETE ==========
Training time : 1.64 minutes

Evaluation samples : 100
Normal             : 53
Attack             : 47

Starting clean VQC prediction...

NSL-KDD SEED 2024 — VQC CLEAN RESULT
Training samples  : 100
Evaluation samples: 100
Accuracy          : 0.69
Macro-F1          : 0.6862
Training time     : 1.64 minutes
Prediction time   : 0.05 minutes

✅ NSL-KDD Seed 2024 VQC completed.


In [15]:
# MILESTONE 4 — VQC — ToN-IoT — Seed 42

import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score

from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit_machine_learning.algorithms import VQC
from qiskit_machine_learning.optimizers import COBYLA

SEED = 42
TRAIN_SAMPLES = 100
EVAL_SAMPLES = 100

# ============================================================
# 1. Load processed ToN-IoT Seed 42
# ============================================================

data = np.load(
    "milestone3_processed/ToN-IoT_seed_42.npz"
)

X_train = data["X_train"]
y_train = data["y_train"]

X_val = data["X_val"]
y_val = data["y_val"]

print("Dataset       : ToN-IoT")
print("Model         : VQC")
print("Seed          :", SEED)
print("Full train    :", X_train.shape)
print("Validation    :", X_val.shape)

# ============================================================
# 2. Stratified training subset
# ============================================================

X_subset, _, y_subset, _ = train_test_split(
    X_train,
    y_train,
    train_size=TRAIN_SAMPLES,
    stratify=y_train,
    random_state=SEED
)

print("\nTraining samples :", len(y_subset))
print("Normal           :", np.sum(y_subset == 0))
print("Attack           :", np.sum(y_subset == 1))

# ============================================================
# 3. Scale features to [0, pi]
# ============================================================

scaler = MinMaxScaler(
    feature_range=(0, np.pi)
)

X_subset_scaled = scaler.fit_transform(
    X_subset
)

# ============================================================
# 4. 8-qubit ZZ feature map
# ============================================================

feature_map = zz_feature_map(
    feature_dimension=8,
    reps=1,
    entanglement="linear"
)

# ============================================================
# 5. Variational ansatz
# ============================================================

ansatz = real_amplitudes(
    num_qubits=8,
    reps=1,
    entanglement="linear"
)

print("\nQubits            : 8")
print("Feature-map reps  : 1")
print("Ansatz reps       : 1")

# ============================================================
# 6. VQC
# ============================================================

optimizer = COBYLA(
    maxiter=30
)

vqc = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=optimizer
)

# ============================================================
# 7. Train
# ============================================================

print("\nStarting VQC training...")

train_start = time.time()

vqc.fit(
    X_subset_scaled,
    y_subset
)

train_time = time.time() - train_start

print("\n========== VQC TRAINING COMPLETE ==========")
print(
    "Training time : {:.2f} minutes".format(
        train_time / 60
    )
)

# ============================================================
# 8. Select clean evaluation samples
# ============================================================

X_eval, _, y_eval, _ = train_test_split(
    X_val,
    y_val,
    train_size=EVAL_SAMPLES,
    stratify=y_val,
    random_state=SEED
)

X_eval_scaled = scaler.transform(
    X_eval
)

print("\nEvaluation samples :", len(y_eval))
print("Normal             :", np.sum(y_eval == 0))
print("Attack             :", np.sum(y_eval == 1))

# ============================================================
# 9. Clean prediction
# ============================================================

print("\nStarting clean VQC prediction...")

pred_start = time.time()

y_pred = vqc.predict(
    X_eval_scaled
)

pred_time = time.time() - pred_start

# ============================================================
# 10. Metrics
# ============================================================

accuracy = accuracy_score(
    y_eval,
    y_pred
)

macro_f1 = f1_score(
    y_eval,
    y_pred,
    average="macro"
)

print("\n==========================================")
print("ToN-IoT SEED 42 — VQC CLEAN RESULT")
print("==========================================")

print("Training samples  :", len(y_subset))
print("Evaluation samples:", len(y_eval))
print("Accuracy          :", round(accuracy, 4))
print("Macro-F1          :", round(macro_f1, 4))
print(
    "Training time     : {:.2f} minutes".format(
        train_time / 60
    )
)
print(
    "Prediction time   : {:.2f} minutes".format(
        pred_time / 60
    )
)

print("\n✅ ToN-IoT Seed 42 VQC completed.")

Dataset       : ToN-IoT
Model         : VQC
Seed          : 42
Full train    : (147730, 8)
Validation    : (63313, 8)

Training samples : 100
Normal           : 24
Attack           : 76

Qubits            : 8
Feature-map reps  : 1
Ansatz reps       : 1

Starting VQC training...

========== VQC TRAINING COMPLETE ==========
Training time : 1.58 minutes

Evaluation samples : 100
Normal             : 24
Attack             : 76

Starting clean VQC prediction...

ToN-IoT SEED 42 — VQC CLEAN RESULT
Training samples  : 100
Evaluation samples: 100
Accuracy          : 0.77
Macro-F1          : 0.7048
Training time     : 1.58 minutes
Prediction time   : 0.08 minutes

✅ ToN-IoT Seed 42 VQC completed.


In [16]:
# MILESTONE 4 — VQC — ToN-IoT — Seed 1337

import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score

from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit_machine_learning.algorithms import VQC
from qiskit_machine_learning.optimizers import COBYLA

SEED = 1337
TRAIN_SAMPLES = 100
EVAL_SAMPLES = 100

# ============================================================
# 1. Load processed ToN-IoT Seed 1337
# ============================================================

data = np.load(
    "milestone3_processed/ToN-IoT_seed_1337.npz"
)

X_train = data["X_train"]
y_train = data["y_train"]

X_val = data["X_val"]
y_val = data["y_val"]

print("Dataset       : ToN-IoT")
print("Model         : VQC")
print("Seed          :", SEED)
print("Full train    :", X_train.shape)
print("Validation    :", X_val.shape)

# ============================================================
# 2. Stratified training subset
# ============================================================

X_subset, _, y_subset, _ = train_test_split(
    X_train,
    y_train,
    train_size=TRAIN_SAMPLES,
    stratify=y_train,
    random_state=SEED
)

print("\nTraining samples :", len(y_subset))
print("Normal           :", np.sum(y_subset == 0))
print("Attack           :", np.sum(y_subset == 1))

# ============================================================
# 3. Scale features to [0, pi]
# ============================================================

scaler = MinMaxScaler(
    feature_range=(0, np.pi)
)

X_subset_scaled = scaler.fit_transform(
    X_subset
)

# ============================================================
# 4. 8-qubit ZZ feature map
# ============================================================

feature_map = zz_feature_map(
    feature_dimension=8,
    reps=1,
    entanglement="linear"
)

# ============================================================
# 5. Variational ansatz
# ============================================================

ansatz = real_amplitudes(
    num_qubits=8,
    reps=1,
    entanglement="linear"
)

print("\nQubits            : 8")
print("Feature-map reps  : 1")
print("Ansatz reps       : 1")

# ============================================================
# 6. VQC
# ============================================================

optimizer = COBYLA(
    maxiter=30
)

vqc = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=optimizer
)

# ============================================================
# 7. Train
# ============================================================

print("\nStarting VQC training...")

train_start = time.time()

vqc.fit(
    X_subset_scaled,
    y_subset
)

train_time = time.time() - train_start

print("\n========== VQC TRAINING COMPLETE ==========")
print(
    "Training time : {:.2f} minutes".format(
        train_time / 60
    )
)

# ============================================================
# 8. Select clean evaluation samples
# ============================================================

X_eval, _, y_eval, _ = train_test_split(
    X_val,
    y_val,
    train_size=EVAL_SAMPLES,
    stratify=y_val,
    random_state=SEED
)

X_eval_scaled = scaler.transform(
    X_eval
)

print("\nEvaluation samples :", len(y_eval))
print("Normal             :", np.sum(y_eval == 0))
print("Attack             :", np.sum(y_eval == 1))

# ============================================================
# 9. Clean prediction
# ============================================================

print("\nStarting clean VQC prediction...")

pred_start = time.time()

y_pred = vqc.predict(
    X_eval_scaled
)

pred_time = time.time() - pred_start

# ============================================================
# 10. Metrics
# ============================================================

accuracy = accuracy_score(
    y_eval,
    y_pred
)

macro_f1 = f1_score(
    y_eval,
    y_pred,
    average="macro"
)

print("\n==========================================")
print("ToN-IoT SEED 1337 — VQC CLEAN RESULT")
print("==========================================")

print("Training samples  :", len(y_subset))
print("Evaluation samples:", len(y_eval))
print("Accuracy          :", round(accuracy, 4))
print("Macro-F1          :", round(macro_f1, 4))
print(
    "Training time     : {:.2f} minutes".format(
        train_time / 60
    )
)
print(
    "Prediction time   : {:.2f} minutes".format(
        pred_time / 60
    )
)

print("\n✅ ToN-IoT Seed 1337 VQC completed.")

Dataset       : ToN-IoT
Model         : VQC
Seed          : 1337
Full train    : (147730, 8)
Validation    : (63313, 8)

Training samples : 100
Normal           : 24
Attack           : 76

Qubits            : 8
Feature-map reps  : 1
Ansatz reps       : 1

Starting VQC training...

========== VQC TRAINING COMPLETE ==========
Training time : 1.65 minutes

Evaluation samples : 100
Normal             : 24
Attack             : 76

Starting clean VQC prediction...

ToN-IoT SEED 1337 — VQC CLEAN RESULT
Training samples  : 100
Evaluation samples: 100
Accuracy          : 0.83
Macro-F1          : 0.7482
Training time     : 1.65 minutes
Prediction time   : 0.05 minutes

✅ ToN-IoT Seed 1337 VQC completed.


In [17]:
# MILESTONE 4 — VQC — ToN-IoT — Seed 2024

import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score

from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit_machine_learning.algorithms import VQC
from qiskit_machine_learning.optimizers import COBYLA

SEED = 2024
TRAIN_SAMPLES = 100
EVAL_SAMPLES = 100

# ============================================================
# 1. Load processed ToN-IoT Seed 2024
# ============================================================

data = np.load(
    "milestone3_processed/ToN-IoT_seed_2024.npz"
)

X_train = data["X_train"]
y_train = data["y_train"]

X_val = data["X_val"]
y_val = data["y_val"]

print("Dataset       : ToN-IoT")
print("Model         : VQC")
print("Seed          :", SEED)
print("Full train    :", X_train.shape)
print("Validation    :", X_val.shape)

# ============================================================
# 2. Stratified training subset
# ============================================================

X_subset, _, y_subset, _ = train_test_split(
    X_train,
    y_train,
    train_size=TRAIN_SAMPLES,
    stratify=y_train,
    random_state=SEED
)

print("\nTraining samples :", len(y_subset))
print("Normal           :", np.sum(y_subset == 0))
print("Attack           :", np.sum(y_subset == 1))

# ============================================================
# 3. Scale features to [0, pi]
# ============================================================

scaler = MinMaxScaler(
    feature_range=(0, np.pi)
)

X_subset_scaled = scaler.fit_transform(
    X_subset
)

# ============================================================
# 4. 8-qubit ZZ feature map
# ============================================================

feature_map = zz_feature_map(
    feature_dimension=8,
    reps=1,
    entanglement="linear"
)

# ============================================================
# 5. Variational ansatz
# ============================================================

ansatz = real_amplitudes(
    num_qubits=8,
    reps=1,
    entanglement="linear"
)

print("\nQubits            : 8")
print("Feature-map reps  : 1")
print("Ansatz reps       : 1")

# ============================================================
# 6. VQC
# ============================================================

optimizer = COBYLA(
    maxiter=30
)

vqc = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=optimizer
)

# ============================================================
# 7. Train
# ============================================================

print("\nStarting VQC training...")

train_start = time.time()

vqc.fit(
    X_subset_scaled,
    y_subset
)

train_time = time.time() - train_start

print("\n========== VQC TRAINING COMPLETE ==========")
print(
    "Training time : {:.2f} minutes".format(
        train_time / 60
    )
)

# ============================================================
# 8. Select clean evaluation samples
# ============================================================

X_eval, _, y_eval, _ = train_test_split(
    X_val,
    y_val,
    train_size=EVAL_SAMPLES,
    stratify=y_val,
    random_state=SEED
)

X_eval_scaled = scaler.transform(
    X_eval
)

print("\nEvaluation samples :", len(y_eval))
print("Normal             :", np.sum(y_eval == 0))
print("Attack             :", np.sum(y_eval == 1))

# ============================================================
# 9. Clean prediction
# ============================================================

print("\nStarting clean VQC prediction...")

pred_start = time.time()

y_pred = vqc.predict(
    X_eval_scaled
)

pred_time = time.time() - pred_start

# ============================================================
# 10. Metrics
# ============================================================

accuracy = accuracy_score(
    y_eval,
    y_pred
)

macro_f1 = f1_score(
    y_eval,
    y_pred,
    average="macro"
)

print("\n==========================================")
print("ToN-IoT SEED 2024 — VQC CLEAN RESULT")
print("==========================================")

print("Training samples  :", len(y_subset))
print("Evaluation samples:", len(y_eval))
print("Accuracy          :", round(accuracy, 4))
print("Macro-F1          :", round(macro_f1, 4))
print(
    "Training time     : {:.2f} minutes".format(
        train_time / 60
    )
)
print(
    "Prediction time   : {:.2f} minutes".format(
        pred_time / 60
    )
)

print("\n✅ ToN-IoT Seed 2024 VQC completed.")

Dataset       : ToN-IoT
Model         : VQC
Seed          : 2024
Full train    : (147730, 8)
Validation    : (63313, 8)

Training samples : 100
Normal           : 24
Attack           : 76

Qubits            : 8
Feature-map reps  : 1
Ansatz reps       : 1

Starting VQC training...

========== VQC TRAINING COMPLETE ==========
Training time : 1.61 minutes

Evaluation samples : 100
Normal             : 24
Attack             : 76

Starting clean VQC prediction...

ToN-IoT SEED 2024 — VQC CLEAN RESULT
Training samples  : 100
Evaluation samples: 100
Accuracy          : 0.76
Macro-F1          : 0.6956
Training time     : 1.61 minutes
Prediction time   : 0.06 minutes

✅ ToN-IoT Seed 2024 VQC completed.


In [18]:
# MILESTONE 4 — QNN — NSL-KDD — Seed 42

import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score

from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.algorithms import NeuralNetworkClassifier
from qiskit_machine_learning.optimizers import COBYLA

SEED = 42
TRAIN_SAMPLES = 100
EVAL_SAMPLES = 100

# ============================================================
# 1. Load processed NSL-KDD Seed 42
# ============================================================

data = np.load(
    "milestone3_processed/NSL-KDD_seed_42.npz"
)

X_train = data["X_train"]
y_train = data["y_train"]

X_val = data["X_val"]
y_val = data["y_val"]

print("Dataset       : NSL-KDD")
print("Model         : QNN")
print("Seed          :", SEED)
print("Full train    :", X_train.shape)
print("Validation    :", X_val.shape)

# ============================================================
# 2. Stratified training subset
# ============================================================

X_subset, _, y_subset, _ = train_test_split(
    X_train,
    y_train,
    train_size=TRAIN_SAMPLES,
    stratify=y_train,
    random_state=SEED
)

print("\nTraining samples :", len(y_subset))
print("Normal           :", np.sum(y_subset == 0))
print("Attack           :", np.sum(y_subset == 1))

# ============================================================
# 3. Scale features to [0, pi]
# ============================================================

scaler = MinMaxScaler(
    feature_range=(0, np.pi)
)

X_subset_scaled = scaler.fit_transform(
    X_subset
)

# ============================================================
# 4. Create 8-qubit feature map
# ============================================================

feature_map = zz_feature_map(
    feature_dimension=8,
    reps=1,
    entanglement="linear"
)

# ============================================================
# 5. Create variational ansatz
# ============================================================

ansatz = real_amplitudes(
    num_qubits=8,
    reps=1,
    entanglement="linear"
)

print("\nQubits            : 8")
print("Feature-map reps  : 1")
print("Ansatz reps       : 1")

# ============================================================
# 6. Combine feature map + ansatz
# ============================================================

qc = feature_map.compose(ansatz)

# ============================================================
# 7. Create EstimatorQNN
# ============================================================

qnn = EstimatorQNN(
    circuit=qc,
    input_params=feature_map.parameters,
    weight_params=ansatz.parameters
)

# ============================================================
# 8. Create QNN classifier
# ============================================================

optimizer = COBYLA(
    maxiter=30
)

classifier = NeuralNetworkClassifier(
    neural_network=qnn,
    optimizer=optimizer
)

# ============================================================
# 9. Train QNN
# ============================================================

print("\nStarting QNN training...")

train_start = time.time()

classifier.fit(
    X_subset_scaled,
    y_subset
)

train_time = time.time() - train_start

print("\n========== QNN TRAINING COMPLETE ==========")
print(
    "Training time : {:.2f} minutes".format(
        train_time / 60
    )
)

# ============================================================
# 10. Select clean evaluation samples
# ============================================================

X_eval, _, y_eval, _ = train_test_split(
    X_val,
    y_val,
    train_size=EVAL_SAMPLES,
    stratify=y_val,
    random_state=SEED
)

X_eval_scaled = scaler.transform(
    X_eval
)

print("\nEvaluation samples :", len(y_eval))
print("Normal             :", np.sum(y_eval == 0))
print("Attack             :", np.sum(y_eval == 1))

# ============================================================
# 11. Clean prediction
# ============================================================

print("\nStarting clean QNN prediction...")

pred_start = time.time()

y_pred = classifier.predict(
    X_eval_scaled
)

pred_time = time.time() - pred_start

# ============================================================
# 12. Metrics
# ============================================================

accuracy = accuracy_score(
    y_eval,
    y_pred
)

macro_f1 = f1_score(
    y_eval,
    y_pred,
    average="macro"
)

print("\n==========================================")
print("NSL-KDD SEED 42 — QNN CLEAN RESULT")
print("==========================================")

print("Training samples  :", len(y_subset))
print("Evaluation samples:", len(y_eval))
print("Accuracy          :", round(accuracy, 4))
print("Macro-F1          :", round(macro_f1, 4))
print(
    "Training time     : {:.2f} minutes".format(
        train_time / 60
    )
)
print(
    "Prediction time   : {:.2f} minutes".format(
        pred_time / 60
    )
)

print("\n✅ NSL-KDD Seed 42 QNN completed.")

Dataset       : NSL-KDD
Model         : QNN
Seed          : 42
Full train    : (105815, 8)
Validation    : (45350, 8)

Training samples : 100
Normal           : 53
Attack           : 47

Qubits            : 8
Feature-map reps  : 1
Ansatz reps       : 1

Starting QNN training...

========== QNN TRAINING COMPLETE ==========
Training time : 0.33 minutes

Evaluation samples : 100
Normal             : 53
Attack             : 47

Starting clean QNN prediction...

NSL-KDD SEED 42 — QNN CLEAN RESULT
Training samples  : 100
Evaluation samples: 100
Accuracy          : 0.38
Macro-F1          : 0.2303
Training time     : 0.33 minutes
Prediction time   : 0.01 minutes

✅ NSL-KDD Seed 42 QNN completed.


In [19]:
# MILESTONE 4 — QNN — NSL-KDD — Seed 1337

import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score

from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.algorithms import NeuralNetworkClassifier
from qiskit_machine_learning.optimizers import COBYLA

SEED = 1337
TRAIN_SAMPLES = 100
EVAL_SAMPLES = 100

# ============================================================
# 1. Load processed NSL-KDD Seed 1337
# ============================================================

data = np.load(
    "milestone3_processed/NSL-KDD_seed_1337.npz"
)

X_train = data["X_train"]
y_train = data["y_train"]

X_val = data["X_val"]
y_val = data["y_val"]

print("Dataset       : NSL-KDD")
print("Model         : QNN")
print("Seed          :", SEED)
print("Full train    :", X_train.shape)
print("Validation    :", X_val.shape)

# ============================================================
# 2. Stratified training subset
# ============================================================

X_subset, _, y_subset, _ = train_test_split(
    X_train,
    y_train,
    train_size=TRAIN_SAMPLES,
    stratify=y_train,
    random_state=SEED
)

print("\nTraining samples :", len(y_subset))
print("Normal           :", np.sum(y_subset == 0))
print("Attack           :", np.sum(y_subset == 1))

# ============================================================
# 3. Scale features to [0, pi]
# ============================================================

scaler = MinMaxScaler(
    feature_range=(0, np.pi)
)

X_subset_scaled = scaler.fit_transform(
    X_subset
)

# ============================================================
# 4. 8-qubit ZZ feature map
# ============================================================

feature_map = zz_feature_map(
    feature_dimension=8,
    reps=1,
    entanglement="linear"
)

# ============================================================
# 5. Variational ansatz
# ============================================================

ansatz = real_amplitudes(
    num_qubits=8,
    reps=1,
    entanglement="linear"
)

print("\nQubits            : 8")
print("Feature-map reps  : 1")
print("Ansatz reps       : 1")

# ============================================================
# 6. Combine feature map + ansatz
# ============================================================

qc = feature_map.compose(ansatz)

# ============================================================
# 7. EstimatorQNN
# ============================================================

qnn = EstimatorQNN(
    circuit=qc,
    input_params=feature_map.parameters,
    weight_params=ansatz.parameters
)

# ============================================================
# 8. QNN classifier
# ============================================================

optimizer = COBYLA(
    maxiter=30
)

classifier = NeuralNetworkClassifier(
    neural_network=qnn,
    optimizer=optimizer
)

# ============================================================
# 9. Train
# ============================================================

print("\nStarting QNN training...")

train_start = time.time()

classifier.fit(
    X_subset_scaled,
    y_subset
)

train_time = time.time() - train_start

print("\n========== QNN TRAINING COMPLETE ==========")
print(
    "Training time : {:.2f} minutes".format(
        train_time / 60
    )
)

# ============================================================
# 10. Clean evaluation samples
# ============================================================

X_eval, _, y_eval, _ = train_test_split(
    X_val,
    y_val,
    train_size=EVAL_SAMPLES,
    stratify=y_val,
    random_state=SEED
)

X_eval_scaled = scaler.transform(
    X_eval
)

print("\nEvaluation samples :", len(y_eval))
print("Normal             :", np.sum(y_eval == 0))
print("Attack             :", np.sum(y_eval == 1))

# ============================================================
# 11. Clean prediction
# ============================================================

print("\nStarting clean QNN prediction...")

pred_start = time.time()

y_pred = classifier.predict(
    X_eval_scaled
)

pred_time = time.time() - pred_start

# ============================================================
# 12. Metrics
# ============================================================

accuracy = accuracy_score(
    y_eval,
    y_pred
)

macro_f1 = f1_score(
    y_eval,
    y_pred,
    average="macro"
)

print("\n==========================================")
print("NSL-KDD SEED 1337 — QNN CLEAN RESULT")
print("==========================================")

print("Training samples  :", len(y_subset))
print("Evaluation samples:", len(y_eval))
print("Accuracy          :", round(accuracy, 4))
print("Macro-F1          :", round(macro_f1, 4))
print(
    "Training time     : {:.2f} minutes".format(
        train_time / 60
    )
)
print(
    "Prediction time   : {:.2f} minutes".format(
        pred_time / 60
    )
)

print("\n✅ NSL-KDD Seed 1337 QNN completed.")

Dataset       : NSL-KDD
Model         : QNN
Seed          : 1337
Full train    : (105815, 8)
Validation    : (45350, 8)

Training samples : 100
Normal           : 53
Attack           : 47

Qubits            : 8
Feature-map reps  : 1
Ansatz reps       : 1

Starting QNN training...

========== QNN TRAINING COMPLETE ==========
Training time : 0.33 minutes

Evaluation samples : 100
Normal             : 53
Attack             : 47

Starting clean QNN prediction...

NSL-KDD SEED 1337 — QNN CLEAN RESULT
Training samples  : 100
Evaluation samples: 100
Accuracy          : 0.32
Macro-F1          : 0.2112
Training time     : 0.33 minutes
Prediction time   : 0.01 minutes

✅ NSL-KDD Seed 1337 QNN completed.


In [20]:
# MILESTONE 4 — QNN — NSL-KDD — Seed 2024

import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score

from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.algorithms import NeuralNetworkClassifier
from qiskit_machine_learning.optimizers import COBYLA

SEED = 2024
TRAIN_SAMPLES = 100
EVAL_SAMPLES = 100

# ============================================================
# 1. Load processed NSL-KDD Seed 2024
# ============================================================

data = np.load(
    "milestone3_processed/NSL-KDD_seed_2024.npz"
)

X_train = data["X_train"]
y_train = data["y_train"]

X_val = data["X_val"]
y_val = data["y_val"]

print("Dataset       : NSL-KDD")
print("Model         : QNN")
print("Seed          :", SEED)
print("Full train    :", X_train.shape)
print("Validation    :", X_val.shape)

# ============================================================
# 2. Stratified training subset
# ============================================================

X_subset, _, y_subset, _ = train_test_split(
    X_train,
    y_train,
    train_size=TRAIN_SAMPLES,
    stratify=y_train,
    random_state=SEED
)

print("\nTraining samples :", len(y_subset))
print("Normal           :", np.sum(y_subset == 0))
print("Attack           :", np.sum(y_subset == 1))

# ============================================================
# 3. Scale features to [0, pi]
# ============================================================

scaler = MinMaxScaler(
    feature_range=(0, np.pi)
)

X_subset_scaled = scaler.fit_transform(
    X_subset
)

# ============================================================
# 4. 8-qubit ZZ feature map
# ============================================================

feature_map = zz_feature_map(
    feature_dimension=8,
    reps=1,
    entanglement="linear"
)

# ============================================================
# 5. Variational ansatz
# ============================================================

ansatz = real_amplitudes(
    num_qubits=8,
    reps=1,
    entanglement="linear"
)

print("\nQubits            : 8")
print("Feature-map reps  : 1")
print("Ansatz reps       : 1")

# ============================================================
# 6. Combine feature map + ansatz
# ============================================================

qc = feature_map.compose(ansatz)

# ============================================================
# 7. EstimatorQNN
# ============================================================

qnn = EstimatorQNN(
    circuit=qc,
    input_params=feature_map.parameters,
    weight_params=ansatz.parameters
)

# ============================================================
# 8. QNN classifier
# ============================================================

optimizer = COBYLA(
    maxiter=30
)

classifier = NeuralNetworkClassifier(
    neural_network=qnn,
    optimizer=optimizer
)

# ============================================================
# 9. Train
# ============================================================

print("\nStarting QNN training...")

train_start = time.time()

classifier.fit(
    X_subset_scaled,
    y_subset
)

train_time = time.time() - train_start

print("\n========== QNN TRAINING COMPLETE ==========")
print(
    "Training time : {:.2f} minutes".format(
        train_time / 60
    )
)

# ============================================================
# 10. Clean evaluation samples
# ============================================================

X_eval, _, y_eval, _ = train_test_split(
    X_val,
    y_val,
    train_size=EVAL_SAMPLES,
    stratify=y_val,
    random_state=SEED
)

X_eval_scaled = scaler.transform(
    X_eval
)

print("\nEvaluation samples :", len(y_eval))
print("Normal             :", np.sum(y_eval == 0))
print("Attack             :", np.sum(y_eval == 1))

# ============================================================
# 11. Clean prediction
# ============================================================

print("\nStarting clean QNN prediction...")

pred_start = time.time()

y_pred = classifier.predict(
    X_eval_scaled
)

pred_time = time.time() - pred_start

# ============================================================
# 12. Metrics
# ============================================================

accuracy = accuracy_score(
    y_eval,
    y_pred
)

macro_f1 = f1_score(
    y_eval,
    y_pred,
    average="macro"
)

print("\n==========================================")
print("NSL-KDD SEED 2024 — QNN CLEAN RESULT")
print("==========================================")

print("Training samples  :", len(y_subset))
print("Evaluation samples:", len(y_eval))
print("Accuracy          :", round(accuracy, 4))
print("Macro-F1          :", round(macro_f1, 4))
print(
    "Training time     : {:.2f} minutes".format(
        train_time / 60
    )
)
print(
    "Prediction time   : {:.2f} minutes".format(
        pred_time / 60
    )
)

print("\n✅ NSL-KDD Seed 2024 QNN completed.")

Dataset       : NSL-KDD
Model         : QNN
Seed          : 2024
Full train    : (105815, 8)
Validation    : (45350, 8)

Training samples : 100
Normal           : 53
Attack           : 47

Qubits            : 8
Feature-map reps  : 1
Ansatz reps       : 1

Starting QNN training...

========== QNN TRAINING COMPLETE ==========
Training time : 0.52 minutes

Evaluation samples : 100
Normal             : 53
Attack             : 47

Starting clean QNN prediction...

NSL-KDD SEED 2024 — QNN CLEAN RESULT
Training samples  : 100
Evaluation samples: 100
Accuracy          : 0.4
Macro-F1          : 0.2469
Training time     : 0.52 minutes
Prediction time   : 0.01 minutes

✅ NSL-KDD Seed 2024 QNN completed.


In [21]:
# MILESTONE 4 — QNN — ToN-IoT — Seed 42

import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score

from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.algorithms import NeuralNetworkClassifier
from qiskit_machine_learning.optimizers import COBYLA

SEED = 42
TRAIN_SAMPLES = 100
EVAL_SAMPLES = 100

# ============================================================
# 1. Load processed ToN-IoT Seed 42
# ============================================================

data = np.load(
    "milestone3_processed/ToN-IoT_seed_42.npz"
)

X_train = data["X_train"]
y_train = data["y_train"]

X_val = data["X_val"]
y_val = data["y_val"]

print("Dataset       : ToN-IoT")
print("Model         : QNN")
print("Seed          :", SEED)
print("Full train    :", X_train.shape)
print("Validation    :", X_val.shape)

# ============================================================
# 2. Stratified training subset
# ============================================================

X_subset, _, y_subset, _ = train_test_split(
    X_train,
    y_train,
    train_size=TRAIN_SAMPLES,
    stratify=y_train,
    random_state=SEED
)

print("\nTraining samples :", len(y_subset))
print("Normal           :", np.sum(y_subset == 0))
print("Attack           :", np.sum(y_subset == 1))

# ============================================================
# 3. Scale features to [0, pi]
# ============================================================

scaler = MinMaxScaler(
    feature_range=(0, np.pi)
)

X_subset_scaled = scaler.fit_transform(
    X_subset
)

# ============================================================
# 4. 8-qubit ZZ feature map
# ============================================================

feature_map = zz_feature_map(
    feature_dimension=8,
    reps=1,
    entanglement="linear"
)

# ============================================================
# 5. Variational ansatz
# ============================================================

ansatz = real_amplitudes(
    num_qubits=8,
    reps=1,
    entanglement="linear"
)

print("\nQubits            : 8")
print("Feature-map reps  : 1")
print("Ansatz reps       : 1")

# ============================================================
# 6. Combine feature map + ansatz
# ============================================================

qc = feature_map.compose(ansatz)

# ============================================================
# 7. EstimatorQNN
# ============================================================

qnn = EstimatorQNN(
    circuit=qc,
    input_params=feature_map.parameters,
    weight_params=ansatz.parameters
)

# ============================================================
# 8. QNN classifier
# ============================================================

optimizer = COBYLA(
    maxiter=30
)

classifier = NeuralNetworkClassifier(
    neural_network=qnn,
    optimizer=optimizer
)

# ============================================================
# 9. Train
# ============================================================

print("\nStarting QNN training...")

train_start = time.time()

classifier.fit(
    X_subset_scaled,
    y_subset
)

train_time = time.time() - train_start

print("\n========== QNN TRAINING COMPLETE ==========")
print(
    "Training time : {:.2f} minutes".format(
        train_time / 60
    )
)

# ============================================================
# 10. Clean evaluation samples
# ============================================================

X_eval, _, y_eval, _ = train_test_split(
    X_val,
    y_val,
    train_size=EVAL_SAMPLES,
    stratify=y_val,
    random_state=SEED
)

X_eval_scaled = scaler.transform(
    X_eval
)

print("\nEvaluation samples :", len(y_eval))
print("Normal             :", np.sum(y_eval == 0))
print("Attack             :", np.sum(y_eval == 1))

# ============================================================
# 11. Clean prediction
# ============================================================

print("\nStarting clean QNN prediction...")

pred_start = time.time()

y_pred = classifier.predict(
    X_eval_scaled
)

pred_time = time.time() - pred_start

# ============================================================
# 12. Metrics
# ============================================================

accuracy = accuracy_score(
    y_eval,
    y_pred
)

macro_f1 = f1_score(
    y_eval,
    y_pred,
    average="macro"
)

print("\n==========================================")
print("ToN-IoT SEED 42 — QNN CLEAN RESULT")
print("==========================================")

print("Training samples  :", len(y_subset))
print("Evaluation samples:", len(y_eval))
print("Accuracy          :", round(accuracy, 4))
print("Macro-F1          :", round(macro_f1, 4))
print(
    "Training time     : {:.2f} minutes".format(
        train_time / 60
    )
)
print(
    "Prediction time   : {:.2f} minutes".format(
        pred_time / 60
    )
)

print("\n✅ ToN-IoT Seed 42 QNN completed.")

Dataset       : ToN-IoT
Model         : QNN
Seed          : 42
Full train    : (147730, 8)
Validation    : (63313, 8)

Training samples : 100
Normal           : 24
Attack           : 76

Qubits            : 8
Feature-map reps  : 1
Ansatz reps       : 1

Starting QNN training...

========== QNN TRAINING COMPLETE ==========
Training time : 0.31 minutes

Evaluation samples : 100
Normal             : 24
Attack             : 76

Starting clean QNN prediction...

ToN-IoT SEED 42 — QNN CLEAN RESULT
Training samples  : 100
Evaluation samples: 100
Accuracy          : 0.7
Macro-F1          : 0.2917
Training time     : 0.31 minutes
Prediction time   : 0.01 minutes

✅ ToN-IoT Seed 42 QNN completed.


In [22]:
# MILESTONE 4 — QNN — ToN-IoT — Seed 1337

import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score

from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.algorithms import NeuralNetworkClassifier
from qiskit_machine_learning.optimizers import COBYLA

SEED = 1337
TRAIN_SAMPLES = 100
EVAL_SAMPLES = 100

# ============================================================
# 1. Load processed ToN-IoT Seed 1337
# ============================================================

data = np.load(
    "milestone3_processed/ToN-IoT_seed_1337.npz"
)

X_train = data["X_train"]
y_train = data["y_train"]

X_val = data["X_val"]
y_val = data["y_val"]

print("Dataset       : ToN-IoT")
print("Model         : QNN")
print("Seed          :", SEED)
print("Full train    :", X_train.shape)
print("Validation    :", X_val.shape)

# ============================================================
# 2. Stratified training subset
# ============================================================

X_subset, _, y_subset, _ = train_test_split(
    X_train,
    y_train,
    train_size=TRAIN_SAMPLES,
    stratify=y_train,
    random_state=SEED
)

print("\nTraining samples :", len(y_subset))
print("Normal           :", np.sum(y_subset == 0))
print("Attack           :", np.sum(y_subset == 1))

# ============================================================
# 3. Scale features to [0, pi]
# ============================================================

scaler = MinMaxScaler(
    feature_range=(0, np.pi)
)

X_subset_scaled = scaler.fit_transform(
    X_subset
)

# ============================================================
# 4. 8-qubit ZZ feature map
# ============================================================

feature_map = zz_feature_map(
    feature_dimension=8,
    reps=1,
    entanglement="linear"
)

# ============================================================
# 5. Variational ansatz
# ============================================================

ansatz = real_amplitudes(
    num_qubits=8,
    reps=1,
    entanglement="linear"
)

print("\nQubits            : 8")
print("Feature-map reps  : 1")
print("Ansatz reps       : 1")

# ============================================================
# 6. Combine feature map + ansatz
# ============================================================

qc = feature_map.compose(ansatz)

# ============================================================
# 7. EstimatorQNN
# ============================================================

qnn = EstimatorQNN(
    circuit=qc,
    input_params=feature_map.parameters,
    weight_params=ansatz.parameters
)

# ============================================================
# 8. QNN classifier
# ============================================================

optimizer = COBYLA(
    maxiter=30
)

classifier = NeuralNetworkClassifier(
    neural_network=qnn,
    optimizer=optimizer
)

# ============================================================
# 9. Train
# ============================================================

print("\nStarting QNN training...")

train_start = time.time()

classifier.fit(
    X_subset_scaled,
    y_subset
)

train_time = time.time() - train_start

print("\n========== QNN TRAINING COMPLETE ==========")
print(
    "Training time : {:.2f} minutes".format(
        train_time / 60
    )
)

# ============================================================
# 10. Clean evaluation samples
# ============================================================

X_eval, _, y_eval, _ = train_test_split(
    X_val,
    y_val,
    train_size=EVAL_SAMPLES,
    stratify=y_val,
    random_state=SEED
)

X_eval_scaled = scaler.transform(
    X_eval
)

print("\nEvaluation samples :", len(y_eval))
print("Normal             :", np.sum(y_eval == 0))
print("Attack             :", np.sum(y_eval == 1))

# ============================================================
# 11. Clean prediction
# ============================================================

print("\nStarting clean QNN prediction...")

pred_start = time.time()

y_pred = classifier.predict(
    X_eval_scaled
)

pred_time = time.time() - pred_start

# ============================================================
# 12. Metrics
# ============================================================

accuracy = accuracy_score(
    y_eval,
    y_pred
)

macro_f1 = f1_score(
    y_eval,
    y_pred,
    average="macro"
)

print("\n==========================================")
print("ToN-IoT SEED 1337 — QNN CLEAN RESULT")
print("==========================================")

print("Training samples  :", len(y_subset))
print("Evaluation samples:", len(y_eval))
print("Accuracy          :", round(accuracy, 4))
print("Macro-F1          :", round(macro_f1, 4))
print(
    "Training time     : {:.2f} minutes".format(
        train_time / 60
    )
)
print(
    "Prediction time   : {:.2f} minutes".format(
        pred_time / 60
    )
)

print("\n✅ ToN-IoT Seed 1337 QNN completed.")

Dataset       : ToN-IoT
Model         : QNN
Seed          : 1337
Full train    : (147730, 8)
Validation    : (63313, 8)

Training samples : 100
Normal           : 24
Attack           : 76

Qubits            : 8
Feature-map reps  : 1
Ansatz reps       : 1

Starting QNN training...

========== QNN TRAINING COMPLETE ==========
Training time : 0.33 minutes

Evaluation samples : 100
Normal             : 24
Attack             : 76

Starting clean QNN prediction...

ToN-IoT SEED 1337 — QNN CLEAN RESULT
Training samples  : 100
Evaluation samples: 100
Accuracy          : 0.55
Macro-F1          : 0.26
Training time     : 0.33 minutes
Prediction time   : 0.01 minutes

✅ ToN-IoT Seed 1337 QNN completed.


In [23]:
# MILESTONE 4 — QNN — ToN-IoT — Seed 2024

import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score

from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.algorithms import NeuralNetworkClassifier
from qiskit_machine_learning.optimizers import COBYLA

SEED = 2024
TRAIN_SAMPLES = 100
EVAL_SAMPLES = 100

# ============================================================
# 1. Load processed ToN-IoT Seed 2024
# ============================================================

data = np.load(
    "milestone3_processed/ToN-IoT_seed_2024.npz"
)

X_train = data["X_train"]
y_train = data["y_train"]

X_val = data["X_val"]
y_val = data["y_val"]

print("Dataset       : ToN-IoT")
print("Model         : QNN")
print("Seed          :", SEED)
print("Full train    :", X_train.shape)
print("Validation    :", X_val.shape)

# ============================================================
# 2. Stratified training subset
# ============================================================

X_subset, _, y_subset, _ = train_test_split(
    X_train,
    y_train,
    train_size=TRAIN_SAMPLES,
    stratify=y_train,
    random_state=SEED
)

print("\nTraining samples :", len(y_subset))
print("Normal           :", np.sum(y_subset == 0))
print("Attack           :", np.sum(y_subset == 1))

# ============================================================
# 3. Scale features to [0, pi]
# ============================================================

scaler = MinMaxScaler(
    feature_range=(0, np.pi)
)

X_subset_scaled = scaler.fit_transform(
    X_subset
)

# ============================================================
# 4. 8-qubit ZZ feature map
# ============================================================

feature_map = zz_feature_map(
    feature_dimension=8,
    reps=1,
    entanglement="linear"
)

# ============================================================
# 5. Variational ansatz
# ============================================================

ansatz = real_amplitudes(
    num_qubits=8,
    reps=1,
    entanglement="linear"
)

print("\nQubits            : 8")
print("Feature-map reps  : 1")
print("Ansatz reps       : 1")

# ============================================================
# 6. Combine circuits
# ============================================================

qc = feature_map.compose(ansatz)

# ============================================================
# 7. Estimator QNN
# ============================================================

qnn = EstimatorQNN(
    circuit=qc,
    input_params=feature_map.parameters,
    weight_params=ansatz.parameters
)

# ============================================================
# 8. QNN classifier
# ============================================================

optimizer = COBYLA(
    maxiter=30
)

classifier = NeuralNetworkClassifier(
    neural_network=qnn,
    optimizer=optimizer
)

# ============================================================
# 9. Train
# ============================================================

print("\nStarting QNN training...")

train_start = time.time()

classifier.fit(
    X_subset_scaled,
    y_subset
)

train_time = time.time() - train_start

print("\n========== QNN TRAINING COMPLETE ==========")
print(
    "Training time : {:.2f} minutes".format(
        train_time / 60
    )
)

# ============================================================
# 10. Clean evaluation samples
# ============================================================

X_eval, _, y_eval, _ = train_test_split(
    X_val,
    y_val,
    train_size=EVAL_SAMPLES,
    stratify=y_val,
    random_state=SEED
)

X_eval_scaled = scaler.transform(
    X_eval
)

print("\nEvaluation samples :", len(y_eval))
print("Normal             :", np.sum(y_eval == 0))
print("Attack             :", np.sum(y_eval == 1))

# ============================================================
# 11. Clean prediction
# ============================================================

print("\nStarting clean QNN prediction...")

pred_start = time.time()

y_pred = classifier.predict(
    X_eval_scaled
)

pred_time = time.time() - pred_start

# ============================================================
# 12. Metrics
# ============================================================

accuracy = accuracy_score(
    y_eval,
    y_pred
)

macro_f1 = f1_score(
    y_eval,
    y_pred,
    average="macro"
)

print("\n==========================================")
print("ToN-IoT SEED 2024 — QNN CLEAN RESULT")
print("==========================================")

print("Training samples  :", len(y_subset))
print("Evaluation samples:", len(y_eval))
print("Accuracy          :", round(accuracy, 4))
print("Macro-F1          :", round(macro_f1, 4))
print(
    "Training time     : {:.2f} minutes".format(
        train_time / 60
    )
)
print(
    "Prediction time   : {:.2f} minutes".format(
        pred_time / 60
    )
)

print("\n✅ ToN-IoT Seed 2024 QNN completed.")
print("🎯 MILESTONE 4 — 18/18 TRAINING RUNS COMPLETE")

Dataset       : ToN-IoT
Model         : QNN
Seed          : 2024
Full train    : (147730, 8)
Validation    : (63313, 8)

Training samples : 100
Normal           : 24
Attack           : 76

Qubits            : 8
Feature-map reps  : 1
Ansatz reps       : 1

Starting QNN training...

========== QNN TRAINING COMPLETE ==========
Training time : 0.35 minutes

Evaluation samples : 100
Normal             : 24
Attack             : 76

Starting clean QNN prediction...

ToN-IoT SEED 2024 — QNN CLEAN RESULT
Training samples  : 100
Evaluation samples: 100
Accuracy          : 0.5
Macro-F1          : 0.2433
Training time     : 0.35 minutes
Prediction time   : 0.01 minutes

✅ ToN-IoT Seed 2024 QNN completed.
🎯 MILESTONE 4 — 18/18 TRAINING RUNS COMPLETE


In [24]:
# MILESTONE 4 — SAVE ALL 18 CLEAN RESULTS

import pandas as pd

results = [

    # =========================
    # NSL-KDD — QSVC
    # =========================
    ["NSL-KDD", "QSVC", 42,   0.9000, 0.8980, 3.05, 5.48],
    ["NSL-KDD", "QSVC", 1337, 0.9000, 0.8974, 2.76, 5.42],
    ["NSL-KDD", "QSVC", 2024, 0.9100, 0.9079, 2.81, 5.41],

    # =========================
    # NSL-KDD — VQC
    # =========================
    ["NSL-KDD", "VQC", 42,   0.6500, 0.6457, 1.63, 0.07],
    ["NSL-KDD", "VQC", 1337, 0.5400, 0.5066, 1.60, 0.05],
    ["NSL-KDD", "VQC", 2024, 0.6900, 0.6862, 1.64, 0.05],

    # =========================
    # NSL-KDD — QNN
    # =========================
    ["NSL-KDD", "QNN", 42,   0.3800, 0.2303, 0.33, 0.01],
    ["NSL-KDD", "QNN", 1337, 0.3200, 0.2112, 0.33, 0.01],
    ["NSL-KDD", "QNN", 2024, 0.4000, 0.2469, 0.52, 0.01],

    # =========================
    # ToN-IoT — QSVC
    # =========================
    ["ToN-IoT", "QSVC", 42,   0.9100, 0.8710, 2.77, 5.37],
    ["ToN-IoT", "QSVC", 1337, 0.9600, 0.9417, 2.73, 5.34],
    ["ToN-IoT", "QSVC", 2024, 0.9300, 0.8926, 2.81, 5.43],

    # =========================
    # ToN-IoT — VQC
    # =========================
    ["ToN-IoT", "VQC", 42,   0.7700, 0.7048, 1.58, 0.08],
    ["ToN-IoT", "VQC", 1337, 0.8300, 0.7482, 1.65, 0.05],
    ["ToN-IoT", "VQC", 2024, 0.7600, 0.6956, 1.61, 0.06],

    # =========================
    # ToN-IoT — QNN
    # =========================
    ["ToN-IoT", "QNN", 42,   0.7000, 0.2917, 0.31, 0.01],
    ["ToN-IoT", "QNN", 1337, 0.5500, 0.2600, 0.33, 0.01],
    ["ToN-IoT", "QNN", 2024, 0.5000, 0.2433, 0.35, 0.01],
]

columns = [
    "Dataset",
    "Model",
    "Seed",
    "Accuracy",
    "Macro_F1",
    "Training_Time_Min",
    "Prediction_Time_Min"
]

clean_results = pd.DataFrame(
    results,
    columns=columns
)

# Save CSV
clean_results.to_csv(
    "milestone4_clean_results.csv",
    index=False
)

print("==========================================")
print("MILESTONE 4 — CLEAN RESULTS")
print("==========================================")

print(clean_results.to_string(index=False))

print("\n==========================================")
print("VERIFICATION")
print("==========================================")

print("Total runs:", len(clean_results))

assert len(clean_results) == 18
assert set(clean_results["Seed"]) == {42, 1337, 2024}
assert set(clean_results["Model"]) == {"QSVC", "VQC", "QNN"}
assert set(clean_results["Dataset"]) == {"NSL-KDD", "ToN-IoT"}

print("✅ 18/18 results verified.")
print("✅ All 3 seeds verified.")
print("✅ All 3 models verified.")
print("✅ Both datasets verified.")

print("\nCSV saved as:")
print("milestone4_clean_results.csv")

MILESTONE 4 — CLEAN RESULTS
Dataset Model  Seed  Accuracy  Macro_F1  Training_Time_Min  Prediction_Time_Min
NSL-KDD  QSVC    42      0.90    0.8980               3.05                 5.48
NSL-KDD  QSVC  1337      0.90    0.8974               2.76                 5.42
NSL-KDD  QSVC  2024      0.91    0.9079               2.81                 5.41
NSL-KDD   VQC    42      0.65    0.6457               1.63                 0.07
NSL-KDD   VQC  1337      0.54    0.5066               1.60                 0.05
NSL-KDD   VQC  2024      0.69    0.6862               1.64                 0.05
NSL-KDD   QNN    42      0.38    0.2303               0.33                 0.01
NSL-KDD   QNN  1337      0.32    0.2112               0.33                 0.01
NSL-KDD   QNN  2024      0.40    0.2469               0.52                 0.01
ToN-IoT  QSVC    42      0.91    0.8710               2.77                 5.37
ToN-IoT  QSVC  1337      0.96    0.9417               2.73                 5.34
ToN-IoT  QSV